In [6]:
# ═══════════════════════════════════════════════
# DR Congo PEP 제재 스크리닝 파이프라인
#
#  [수집]   1. OpenSanctions /pep/ → DR Congo 의원 명단
#  [대조]   2. 제재명단 대조 (ID 확정 + 어순 무관 이름·별칭 + 생년월일)
#           3. 감시 대상 선정 + 인물 프로필 (직위·나이·인적사항·제재 상세)
#  [제재]   4. 스크리닝 결과 저장
#           5. 스크리닝 요약
#           6. 제재 출처·조치 라벨링, A/B/C 카테고리
#  [뉴스]   7. 구글 뉴스 RSS 검색 준비 (두 어순)
#           8. 새 기사 감지·저장 (LOOP=True면 자동 반복)
#  [기사]   9. 최신 헤드라인 5건 + 한글 번역
#           9-2. 셀레니움으로 기사 원문 페이지 확인 (리드·발행일, robots.txt 준수)
#          10. 전체 기사 분류 (행위 유형·역할) + 한줄요약
#  [위험]  11. AHP 기본 치명도 + 기사 기반 가중치 조정
#  [완료]  파이프라인 요약 (기사 수 반영 요약 저장)
#
#  실행: 노트북에서 %run drc_sanction_monitor.py
# ═══════════════════════════════════════════════
import sys, subprocess, importlib
try:
    import feedparser
except ImportError:                                   # 없으면 현재 환경에 자동 설치
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "feedparser"])
    importlib.invalidate_caches()
    import feedparser

import os, re, time, unicodedata
from collections import defaultdict
from datetime import datetime
from urllib.parse import urljoin, quote
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.opensanctions.org"
HEADERS = {"User-Agent": "Mozilla/5.0 (study project)"}
COUNTRY = "DR Congo"
PREFIX = "drc"
EXPECTED = 561

LOOP = False                  # True: 30분마다 자동 업데이트 / False: 한 번만 실행
INTERVAL = 30 * 60
WATCH_EXTRA = []              # 감시에 직접 추가할 이름 (예: ["Felix Tshisekedi"])

# 10단계 기사 분류·한줄요약 방식: False면 키워드 규칙, True면 Claude API(ANTHROPIC_API_KEY 필요)
USE_LLM = False
LLM_MODEL = "claude-haiku-5-5"

# 9-2단계 셀레니움: 구글 뉴스 링크는 자바스크립트로 언론사에 이동하므로 실제 브라우저로 열어
#                   언론사가 공개한 요약(meta description)·제목·발행일만 확인 (본문 전체는 수집 안 함)
USE_SELENIUM = True
SELENIUM_HEADLESS = True          # False면 브라우저 창이 보임
SELENIUM_MAX_PER_PERSON = 5       # 인물당 확인할 최신 기사 수

# 3단계 인물 프로필에 쓸 OpenSanctions 상세 파일(FtM) — 처음 한 번 받아 ftm_cache/에 저장
PROFILE_DATASETS = ["cd_national_assembly",        # 의원 명단: 직위·재직 기간·인적사항
                    "un_sc_sanctions",             # UN 제재: 지정 사유·직함·비고
                    "eu_fsf",                      # EU 금융제재: 직함·비고
                    "gb_fcdo_sanctions"]           # 영국 제재
# 제재 카테고리 판단용 기관 가정 (한국 금융기관 기준 예시 — 실제 기관 상황에 맞게 수정)
INSTITUTION = {"eu_nexus": False,   # EU 지점·EU 거래 연결고리가 있는가
               "uk_nexus": False,   # 영국 연결고리
               "usd_nexus": True}   # 달러 결제가 있는가 (OFAC 영향)


def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))


def tokens(s):
    return [t for t in re.split(r"[\s\-,.']+", strip_accents(s).lower()) if len(t) > 1]


# ───────────────────────────────────────────────
# 1단계: /pep/ 에서 DR Congo 데이터셋 찾기 → CSV 받기
# ───────────────────────────────────────────────
res = requests.get(f"{BASE}/pep/", headers=HEADERS, timeout=10)
print("상태 코드:", res.status_code)
if res.status_code != 200 or COUNTRY not in res.text:
    raise SystemExit("페이지를 못 받았거나 국가 이름이 없어요.")

soup = BeautifulSoup(res.text, "html.parser")
sources = {}
for a in soup.find_all("a", href=True):
    href, text = a["href"], a.get_text(strip=True)
    if href.startswith("/datasets/") and COUNTRY in text:
        sources[href.strip("/").split("/")[-1]] = text

print(f"\n{COUNTRY} 데이터셋 {len(sources)}개")
for slug, name in sources.items():
    print(f" - {name} ({slug})")
if not sources:
    raise SystemExit("데이터셋 링크를 못 찾았어요.")

frames = []
for slug, name in sources.items():
    time.sleep(10)                                   # robots.txt Crawl-delay 준수
    ds_url = f"{BASE}/datasets/{slug}/"
    ds_soup = BeautifulSoup(requests.get(ds_url, headers=HEADERS, timeout=10).text, "html.parser")
    links = [urljoin(ds_url, a["href"]) for a in ds_soup.find_all("a", href=True) if ".csv" in a["href"]]
    csv_url = links[0] if links else f"https://data.opensanctions.org/datasets/latest/{slug}/targets.simple.csv"
    df_src = pd.read_csv(csv_url)
    df_src["source"] = name
    frames.append(df_src)
    print(f"✅ {name}: {len(df_src)}행")

peps = pd.concat(frames, ignore_index=True)
if "schema" in peps:
    peps = peps[peps["schema"] == "Person"]
peps = peps.drop_duplicates("id").reset_index(drop=True)
print(f"\n인물 {len(peps)}명 (예상 약 {EXPECTED}명)")

# ───────────────────────────────────────────────
# 2단계: 제재명단 대조 (ID 확정 + 어순 무관 이름·별칭 후보)
# ───────────────────────────────────────────────
SANC_URL = "https://data.opensanctions.org/datasets/latest/sanctions/targets.simple.csv"
SANC_FILE = "sanctions_cache_v2.csv"
keep = {"id", "schema", "name", "aliases", "birth_date", "countries", "sanctions", "dataset"}

if os.path.exists(SANC_FILE):
    sanc = pd.read_csv(SANC_FILE, low_memory=False)
else:
    print("\n제재명단 다운로드 중... (처음 한 번만)")
    sanc = pd.read_csv(SANC_URL, usecols=lambda c: c in keep, low_memory=False)
    sanc.to_csv(SANC_FILE, index=False)
print(f"제재명단 {len(sanc):,}건")

id_hits = peps.merge(sanc, on="id", how="inner", suffixes=("", "_sanc"))
if "dataset_sanc" not in id_hits.columns and "dataset" in id_hits.columns:   # 명단에 dataset 열이 없을 때 대비
    id_hits = id_hits.rename(columns={"dataset": "dataset_sanc"})

index = defaultdict(list)
for r in sanc[sanc["schema"] == "Person"].to_dict("records"):
    names = [r["name"]] + (str(r["aliases"]).split(";") if pd.notna(r.get("aliases")) else [])
    for n in names:
        tk = frozenset(tokens(n))
        if len(tk) >= 2:
            for t in tk:
                index[t].append((tk, r["id"], n, r.get("birth_date"), r.get("dataset"), r.get("sanctions")))

rows = []
for p in peps.to_dict("records"):
    ptk, seen_ids = set(tokens(p["name"])), set()
    pbd = str(p.get("birth_date"))[:10]
    for t in ptk:
        for tk, sid, sname, sbd, sds, sprog in index.get(t, []):
            if sid in seen_ids or not tk <= ptk:
                continue
            seen_ids.add(sid)
            rows.append({"pep_id": p["id"], "pep_name": p["name"], "pep_birth": p.get("birth_date"),
                         "sanc_id": sid, "matched_name": sname, "sanc_birth": sbd,
                         "birth_match": bool(pd.notna(sbd) and pbd != "nan" and str(sbd)[:10] == pbd),
                         "sanc_dataset": sds, "sanc_programs": sprog})

cands = pd.DataFrame(rows, columns=["pep_id", "pep_name", "pep_birth", "sanc_id",
                                    "matched_name", "sanc_birth", "birth_match", "sanc_dataset",
                                    "sanc_programs"])
cands = cands[~cands["pep_id"].isin(id_hits["id"])].copy()
cands["verdict"] = cands["birth_match"].map(
    {True: "동일인 가능성 높음 (생년월일 일치)", False: "오탐 가능성 (생년월일 불일치·없음)"})

print(f"\n[제재 ID 일치 — 확정] {len(id_hits)}명")
if len(id_hits):
    cols = [c for c in ["name", "birth_date", "sanctions", "dataset_sanc"] if c in id_hits.columns]
    print(id_hits[cols].to_string(index=False))
print(f"\n[이름·별칭 일치 — 후보] {len(cands)}건 (생년월일 일치 {int(cands['birth_match'].sum())}건)")
if len(cands):
    print(cands[["pep_name", "matched_name", "verdict"]].to_string(index=False))

# ───────────────────────────────────────────────
# 3단계: 감시 대상 정하기 (확정 → 생년월일 일치 후보 → 후보 전체)
# ───────────────────────────────────────────────
watch = list(id_hits["name"]) + cands.loc[cands["birth_match"], "pep_name"].tolist()
if not watch:
    watch = cands["pep_name"].tolist()
watch = list(dict.fromkeys(watch + WATCH_EXTRA))

print(f"\n감시 대상 {len(watch)}명")
for n in watch:
    print(" -", n)
if not watch:
    raise SystemExit("감시할 제재 인물이 없어요. WATCH_EXTRA에 이름을 직접 넣어보세요.")


# ───────────────────────────────────────────────
# 3-2단계: 감시 대상 인물 프로필 — 직위·나이·인적사항·제재 상세 (OpenSanctions FtM)
# ───────────────────────────────────────────────
import json
from IPython.display import display
pd.set_option("display.max_colwidth", None)

os.makedirs("ftm_cache", exist_ok=True)
watch_ids = set(id_hits["id"]) | set(cands.loc[cands["birth_match"], "pep_id"]) \
          | set(cands.loc[cands["birth_match"], "sanc_id"])
id_to_person = {**{r["id"]: r["name"] for _, r in id_hits.iterrows()},
                **{r["pep_id"]: r["pep_name"] for _, r in cands[cands["birth_match"]].iterrows()},
                **{r["sanc_id"]: r["pep_name"] for _, r in cands[cands["birth_match"]].iterrows()}}
watch_tokens = {n: set(tokens(n)) for n in watch}


def ftm_lines(ds):
    """데이터셋 FtM 파일을 한 번만 받아 캐시하고 한 줄씩 돌려줌"""
    path = os.path.join("ftm_cache", f"{ds}.ftm.json")
    if not os.path.exists(path):
        url = f"https://data.opensanctions.org/datasets/latest/{ds}/entities.ftm.json"
        print(f"  {ds} 상세 파일 다운로드 중...")
        tmp = path + ".part"                              # 받다가 끊겨도 깨진 캐시가 남지 않게 임시 파일 사용
        try:
            with requests.get(url, headers=HEADERS, stream=True, timeout=120) as r:
                if r.status_code != 200:
                    print(f"  ⚠ {ds}: 상태 {r.status_code} — 건너뜀")
                    return
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            os.replace(tmp, path)
        except Exception as e:
            if os.path.exists(tmp):
                os.remove(tmp)
            print(f"  ⚠ {ds}: 다운로드 실패 ({type(e).__name__}) — 건너뜀")
            return
        time.sleep(2)
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def match_person(e):
    """FtM 인물이 감시 대상 누구인지: ID 우선, 없으면 이름 단어 일치 + 생년월일 일치"""
    if e["id"] in id_to_person:
        return id_to_person[e["id"]]
    props = e.get("properties", {})
    names = props.get("name", []) + props.get("alias", [])
    for person, tk in watch_tokens.items():
        if any(len(set(tokens(n)) & tk) >= 2 for n in names):
            pbd = str(peps.loc[peps["name"] == person, "birth_date"].iloc[0])[:10] \
                if (peps["name"] == person).any() else ""
            if not props.get("birthDate") or pbd[:4] in " ".join(props.get("birthDate", [])):
                return person
    return None


prof = {n: {"직위": set(), "재직": [], "출생지": set(), "성별": set(), "국적": set(),
            "별칭": set(), "제재 기록상 직함·비고": set(), "제재 상세": [], "출처": set()} for n in watch}
positions, occupancies, sanctions_ftm, ent_person = {}, [], [], {}

print("\n" + "═" * 60)
print("3-2단계: 감시 대상 인물 프로필 (OpenSanctions 상세 데이터)")
print("═" * 60)
for ds in PROFILE_DATASETS:
    for e in ftm_lines(ds) or []:
        sch, pr = e.get("schema"), e.get("properties", {})
        if sch == "Position":
            positions[e["id"]] = pr.get("name", [""])[0]
        elif sch == "Occupancy":
            occupancies.append(pr)
        elif sch == "Sanction":
            sanctions_ftm.append((ds, pr))
        elif sch == "Person":
            person = match_person(e)
            if person:
                ent_person[e["id"]] = person
                d = prof[person]
                d["출처"].add(ds)
                d["출생지"].update(pr.get("birthPlace", []))
                d["성별"].update(pr.get("gender", []))
                d["국적"].update(pr.get("nationality", []) + pr.get("citizenship", []))
                d["별칭"].update(pr.get("alias", []))
                d["직위"].update(pr.get("position", []))
                if ds != "cd_national_assembly":
                    d["제재 기록상 직함·비고"].update(pr.get("title", []) + pr.get("position", [])
                                                 + pr.get("notes", [])[:2])

for oc in occupancies:                                  # 재직 기록: 직위·기간·상태
    for h in oc.get("holder", []):
        if h in ent_person:
            post = "; ".join(positions.get(p, p) for p in oc.get("post", []))
            prof[ent_person[h]]["재직"].append(
                f"{post} ({'; '.join(oc.get('startDate', ['?']))} ~ {'; '.join(oc.get('endDate', ['현재']))}"
                f"{', ' + oc['status'][0] if oc.get('status') else ''})")
for ds, sp in sanctions_ftm:                            # 제재 상세: 기관·프로그램·사유·지정일
    for ent in sp.get("entity", []):
        if ent in ent_person:
            prof[ent_person[ent]]["제재 상세"].append(
                {"데이터셋": ds, "기관": "; ".join(sp.get("authority", [])),
                 "프로그램": "; ".join(sp.get("program", [])),
                 "지정 사유": "; ".join(sp.get("reason", []))[:300],
                 "지정일": "; ".join(sp.get("listingDate", []) or sp.get("startDate", []))})

today = pd.Timestamp.today()
profile_rows = []
for person in watch:
    d = prof[person]
    rec = peps[peps["name"] == person]
    bd = str(rec["birth_date"].iloc[0])[:10] if len(rec) and pd.notna(rec["birth_date"].iloc[0]) else ""
    bdt = pd.to_datetime(bd, errors="coerce")
    age = int((today - bdt).days // 365.25) if pd.notna(bdt) else None
    reasons = [x["지정 사유"] for x in d["제재 상세"] if x["지정 사유"]]
    profile_rows.append({
        "인물": person,
        "생년월일": bd or "정보 없음", "나이": age if age is not None else "정보 없음",
        "성별": "; ".join(sorted(d["성별"])) or "정보 없음",
        "출생지": "; ".join(sorted(d["출생지"])) or "정보 없음",
        "국적": "; ".join(sorted(d["국적"])) or "정보 없음",
        "현 직위·재직": " | ".join(d["재직"]) or "; ".join(sorted(d["직위"])) or "정보 없음",
        "제재 기록상 직함·비고": " | ".join(sorted(d["제재 기록상 직함·비고"]))[:300] or "정보 없음",
        "별칭": "; ".join(sorted(d["별칭"]))[:150] or "없음",
        "지정 사유": " | ".join(reasons)[:400] or "공식 데이터에 사유 없음 (원 공시 확인)",
        "출처 데이터셋": ", ".join(sorted(d["출처"])) or "없음",
    })
profile_df = pd.DataFrame(profile_rows)
SANC_REASON = {r["인물"]: r["지정 사유"] for r in profile_rows
               if not r["지정 사유"].startswith("공식 데이터에")}

for r in profile_rows:
    print(f"\n■ {r['인물']}")
    print(f"  생년월일·나이: {r['생년월일']} ({r['나이']}세)  ·  성별: {r['성별']}  ·  출생지: {r['출생지']}  ·  국적: {r['국적']}")
    print(f"  직위·재직: {r['현 직위·재직']}")
    print(f"  제재 기록상 직함·비고: {r['제재 기록상 직함·비고']}")
    print(f"  지정 사유: {r['지정 사유']}")
    print(f"  출처: {r['출처 데이터셋']}")
detail_rows = [{"인물": n, **x} for n in watch for x in prof[n]["제재 상세"]]
if detail_rows:
    print("\n[제재 상세 기록]")
    display(pd.DataFrame(detail_rows))
profile_df.to_csv(f"{PREFIX}_person_profile.csv", index=False, encoding="utf-8-sig")
print(f"\n✅ 저장: {PREFIX}_person_profile.csv  (공식 데이터에 있는 항목만 표시, 없는 항목은 '정보 없음')")


# ───────────────────────────────────────────────
# 4단계: 스크리닝 결과 저장 (분석용, 재배포 금지)
# ───────────────────────────────────────────────
peps.to_csv(f"{PREFIX}_peps.csv", index=False, encoding="utf-8-sig")
id_hits.to_csv(f"{PREFIX}_sanction_id_hits.csv", index=False, encoding="utf-8-sig")
cands.to_csv(f"{PREFIX}_sanction_candidates.csv", index=False, encoding="utf-8-sig")

# ───────────────────────────────────────────────
# 5단계: 스크리닝 요약 (기사 수는 파이프라인 마지막 요약에서 반영)
# ───────────────────────────────────────────────
def n_lists(x):
    return len(str(x).split(";")) if pd.notna(x) else 0

summary = []
for _, r in id_hits.iterrows():
    summary.append({"인물": r["name"], "판정": "확정 (ID 일치)",
                    "제재 목록 수": n_lists(r.get("dataset_sanc")), "제재 목록": r.get("dataset_sanc")})
for _, r in cands.iterrows():
    summary.append({"인물": r["pep_name"], "판정": r["verdict"],
                    "제재 목록 수": n_lists(r["sanc_dataset"]), "제재 목록": r["sanc_dataset"]})
summary = pd.DataFrame(summary)

if len(summary):
    summary.to_csv(f"{PREFIX}_summary.csv", index=False, encoding="utf-8-sig")

print("\n" + "═" * 60)
print(f"요약: {COUNTRY} 의원 {len(peps)}명 중")
print(f"  · 제재 확정 {len(id_hits)}명")
print(f"  · 동일인 가능성 높은 후보 {int(cands['birth_match'].sum())}명")
print(f"  · 오탐으로 판단한 후보 {int((~cands['birth_match']).sum())}명")
if len(summary):
    print(summary[["인물", "판정", "제재 목록 수"]].to_string(index=False))
print("═" * 60)
print("\n✅ 스크리닝 결과 저장 완료")

# ───────────────────────────────────────────────
# 6단계: 제재 출처·명목·카테고리(A 금지 / B 제한 / C 감시) 라벨링
# ───────────────────────────────────────────────
from IPython.display import display
pd.set_option("display.max_colwidth", None)

# 목록 이름 → (원지정 기관, 독자/이행, 조치 종류)
LIST_RULES = [
    ("un security council",    "UN 안보리",   "원지정", "자산동결·여행금지"),
    ("ofac",                   "미국 OFAC",   "원지정", "자산동결"),
    ("eu consolidated travel", "EU",          "원지정", "여행금지"),
    ("eu council official",    "EU",          "원지정", "법적 지정 근거(관보)"),
    ("eu financial sanctions", "EU",          "원지정", "자산동결"),
    ("uk fcdo",                "영국",        "원지정", "자산동결"),
    ("belgian",                "EU",          "이행(벨기에)", "자산동결"),
    ("french",                 "EU",          "이행(프랑스)", "자산동결"),
    ("monaco",                 "EU",          "이행(모나코)", "자산동결"),
    ("swiss",                  "EU",          "이행(스위스)", "자산동결"),
    ("australian",             "UN 안보리",   "이행(호주)", "자산동결"),
    ("japan",                  "UN 안보리",   "이행(일본)", "자산동결"),
    ("south africa",           "UN 안보리",   "이행(남아공)", "자산동결"),
    ("argentina",              "UN 안보리",   "이행(아르헨티나)", "자산동결"),
    ("strategic high-tech",    "대만",        "독자", "수출통제"),
    ("sfms blacklist",         "우크라이나",  "독자", "FIU 감시목록"),
    ("iran",                   "이란",        "독자", "자산동결"),
]

def label_list(lst):
    s_ = " " + lst.lower() + " "
    for key, origin, kind, measure in LIST_RULES:
        if key in s_:
            return origin, kind, measure
    return "기타", "독자", "확인 필요"

def category_of(origin, measure):
    """우리 기관(INSTITUTION 가정)에 대한 구속력과 조치 종류로 A/B/C 판정"""
    freeze = "자산동결" in measure
    binding = (origin == "UN 안보리"
               or (origin == "미국 OFAC" and INSTITUTION["usd_nexus"])
               or (origin == "EU" and INSTITUTION["eu_nexus"])
               or (origin == "영국" and INSTITUTION["uk_nexus"]))
    if binding and freeze:
        return "A", "구속력 있는 자산동결 → 거래 금지·동결·보고 (점수 계산 없이 오버라이드)"
    if freeze or measure in ("수출통제",):
        return "B", "구속력 없는 자산동결 또는 수출통제 → EDD·상급자 승인·거래 제한"
    return "C", "여행금지·감시목록 등 → 위험 요인으로 반영, 모니터링 강화"

def program_of(text):
    """OpenSanctions 'sanctions' 필드에서 제재 명목(프로그램) 요약"""
    if pd.isna(text) or not str(text).strip():
        return "명목 정보 없음 (원 공시 확인 필요)"
    t = str(text)
    tags = []
    for key, tag in [("congo", "DR Congo 제재 프로그램"), ("terror", "테러"),
                     ("iran", "이란 제재"), ("human rights", "인권 침해"),
                     ("arms", "무기 금수 위반"), ("armed group", "무장단체"),
                     ("corruption", "부패")]:
        if key in t.lower() and tag not in tags:
            tags.append(tag)
    head = t.replace(";", " / ")[:160]
    return (", ".join(tags) + " | " if tags else "") + head

# 대상: 확정 + 동일인 후보 (오탐 제외)
targets = []
prog_col = "sanctions_sanc" if "sanctions_sanc" in id_hits.columns else "sanctions"
for _, r in id_hits.iterrows():
    targets.append((r["name"], "확정", r.get("dataset_sanc"), r.get(prog_col)))
for _, r in cands[cands["birth_match"]].iterrows():
    targets.append((r["pep_name"], "동일인 후보", r["sanc_dataset"], r.get("sanc_programs")))

# 명목이 비어 있으면 3-2단계에서 찾은 FtM 지정 사유로 채움
targets = [(p, v, l, pr if pd.notna(pr) and str(pr).strip() else SANC_REASON.get(p)) for p, v, l, pr in targets]

RANK = {"A": 0, "B": 1, "C": 2}
lab_rows = []
for person, verdict, lists, progs in targets:
    for lst in str(lists).split(";"):
        lst = lst.strip()
        if not lst or lst == "nan":
            continue
        origin, kind, measure = label_list(lst)
        cat, why = category_of(origin, measure)
        lab_rows.append({"인물": person, "판정": verdict, "제재 목록": lst,
                         "원지정 기관": origin, "구분": kind, "조치": measure,
                         "카테고리": cat, "근거": why})
labeled = pd.DataFrame(lab_rows)

cat_rows = []
if len(labeled):
    labeled["_r"] = labeled["카테고리"].map(RANK)
    labeled = labeled.sort_values(["인물", "_r"]).drop(columns="_r")

    print("\n" + "═" * 60)
    print("제재 출처·명목·카테고리")
    print(f"기관 가정: EU 연결 {INSTITUTION['eu_nexus']}, 영국 연결 {INSTITUTION['uk_nexus']}, "
          f"달러 결제 {INSTITUTION['usd_nexus']}")
    print("═" * 60)
    display(labeled[["인물", "제재 목록", "원지정 기관", "구분", "조치", "카테고리"]].reset_index(drop=True))

    for person, verdict, lists, progs in targets:
        g = labeled[labeled["인물"] == person]
        if g.empty:
            continue
        top = g.iloc[0]
        origins = sorted(set(g["원지정 기관"]) - {"대만", "우크라이나"})
        cat_rows.append({"인물": person, "판정": verdict,
                         "최종 카테고리": top["카테고리"],
                         "가장 강한 제재": f"{top['원지정 기관']} · {top['조치']}",
                         "독립 원지정 기관": ", ".join(origins),
                         "목록 수": len(g),
                         "제재 명목": program_of(progs),
                         "처리 방식": top["근거"]})
    cat_df = pd.DataFrame(cat_rows).sort_values("최종 카테고리")
    print("\n[인물별 최종 카테고리]")
    display(cat_df.reset_index(drop=True))
    for _, r in cat_df.iterrows():
        print(f"\n■ {r['인물']} → {r['최종 카테고리']}급")
        print(f"  가장 강한 제재: {r['가장 강한 제재']} (독립 원지정: {r['독립 원지정 기관']}, 목록 {r['목록 수']}개)")
        print(f"  명목: {r['제재 명목']}")
        print(f"  처리: {r['처리 방식']}")
    labeled.to_csv(f"{PREFIX}_sanction_labeled.csv", index=False, encoding="utf-8-sig")
    cat_df.to_csv(f"{PREFIX}_sanction_category.csv", index=False, encoding="utf-8-sig")
    print(f"\n✅ 저장: {PREFIX}_sanction_labeled.csv, {PREFIX}_sanction_category.csv")
    print("   ※ 카테고리 기준과 기관 가정은 설명용 예시이며, 실제 적용은 기관 정책·법적 검토가 필요해요.")

# ───────────────────────────────────────────────
# 7단계: 뉴스 검색 준비 — 인물 이름으로 구글 뉴스 RSS (프랑스어·콩고판)
# ───────────────────────────────────────────────
def short(n):                                        # 후성·부칭 빼고 이름+성
    t = n.split()
    return f"{t[0]} {t[-1]}" if len(t) >= 3 else n


def query_for(n):
    # 콩고 공식 명단은 "성 후성 이름" 순서, 기사는 보통 "이름 성" 순서 → 두 순서 모두 검색
    t = n.split()
    a, b = (t[0], t[-1]) if len(t) >= 2 else (n, "")
    return f'"{b} {a}" OR "{a} {b}"' if b else f'"{a}"'


FEED_PERSON = {}
for n in watch:
    q = quote(query_for(n))
    FEED_PERSON[f"https://news.google.com/rss/search?q={q}&hl=fr&gl=CD&ceid=CD:fr"] = n


def fetch_person_news():
    items = []
    for url, person in FEED_PERSON.items():
        # requests로 받아서(리다이렉트 자동 추적) feedparser에 넘김
        try:
            r = requests.get(url, headers=HEADERS, timeout=20, allow_redirects=True)
        except Exception as e:
            print(f"  ⚠ {person}: 요청 실패 ({e})")
            continue
        feed = feedparser.parse(r.content)
        if r.status_code != 200 or not feed.entries:
            print(f"  ⚠ {person}: 상태 {r.status_code}, 기사 {len(feed.entries)}건, 최종 주소 {r.url[:80]}")
            if "consent" in r.url:
                print("     → 구글 동의(consent) 페이지로 이동됨: 브라우저에서 위 주소를 한 번 열어보세요")
        print(f"  {person}: {len(feed.entries)}건")
        for e in feed.entries:
            items.append({"person": person,
                          "title": e.get("title"),
                          "source": e.get("source", {}).get("title", ""),
                          "published": e.get("published", "")[:25],
                          "url": e.get("link")})
        time.sleep(3)                                # 인물별 요청 간격
    return items


# ───────────────────────────────────────────────
# 8단계: 뉴스 수집 — 새 기사만 표시·저장 (LOOP=True면 자동 반복)
# ───────────────────────────────────────────────
OUT = f"{PREFIX}_sanction_news_monitor.csv"
seen = set(pd.read_csv(OUT)["url"]) if os.path.exists(OUT) else set()
first_run = not seen

while True:
    print(f"\n[{datetime.now():%Y-%m-%d %H:%M}] {len(watch)}명 뉴스 확인 중...")
    articles = fetch_person_news()
    new = [a for a in articles if a["url"] and a["url"] not in seen]
    for a in new:
        seen.add(a["url"])
        a["found_at"] = f"{datetime.now():%Y-%m-%d %H:%M}"

    label = "기존 기사(첫 실행)" if first_run else "새 기사"
    print(f"  받은 기사 {len(articles)}건 · {label} {len(new)}건")
    for a in sorted(new, key=lambda x: x["person"])[:30]:
        print(f"  🔔 {a['person']} | {a['title']} ({a['source']}, {a['published'][:16]})")

    if new:
        pd.DataFrame(new).to_csv(OUT, mode="a", index=False,
                                 header=not os.path.exists(OUT), encoding="utf-8-sig")
    first_run = False
    if not LOOP:
        break
    time.sleep(INTERVAL)

# ───────────────────────────────────────────────
# 9단계: 인물별 최신 기사 5건 — 원문 헤드라인 + 한글 번역
# ───────────────────────────────────────────────
try:
    import deep_translator
except ImportError:
    print("\n번역 라이브러리(deep-translator) 설치 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deep-translator"])
    importlib.invalidate_caches()
from deep_translator import GoogleTranslator, MyMemoryTranslator
from IPython.display import display

pd.set_option("display.max_colwidth", None)

# 번역 엔진 선택: 구글 → 안 되면 MyMemory → 둘 다 안 되면 번역 생략
TEST = "Bonjour, comment allez-vous ?"
ENGINES = [("Google", lambda: GoogleTranslator(source="auto", target="ko")),
           ("MyMemory", lambda: MyMemoryTranslator(source="fr-FR", target="ko-KR"))]
translator, engine_name = None, None
for name, make in ENGINES:
    try:
        t = make()
        out = t.translate(TEST)
        if out and out.strip() != TEST:
            translator, engine_name = t, name
            print(f"\n번역 엔진: {name} (테스트: '{TEST}' → '{out}')")
            break
    except Exception as e:
        print(f"\n⚠ {name} 번역 실패: {type(e).__name__} {str(e)[:100]}")
if translator is None:
    print("⚠ 사용 가능한 번역 엔진이 없어 번역 없이 원문만 표시해요.")


def to_korean(text):
    if translator is None:
        return "(번역 불가)"
    try:
        result = translator.translate(str(text))
        time.sleep(0.5)                               # 번역 요청 간격
        return result
    except Exception as e:
        return f"(번역 실패: {type(e).__name__})"


def clean_headline(title, source):
    # 구글 뉴스 제목 끝의 " - 언론사명" 제거
    t, s = str(title), str(source)
    return t.rsplit(" - ", 1)[0] if s and t.endswith(s) else t


news = pd.read_csv(OUT)
# 요일 접두어("Tue, ")를 떼고 고정 형식으로 파싱 — 요일·형식 추론 오류로 날짜가 비는 문제 방지
_pub = news["published"].astype(str).str.replace(r"^[A-Za-z]{3},\s*", "", regex=True).str[:20]
news["date"] = pd.to_datetime(_pub, format="%d %b %Y %H:%M:%S", errors="coerce", utc=True)
news["date"] = news["date"].fillna(pd.to_datetime(news["published"], errors="coerce", utc=True)).dt.strftime("%Y-%m-%d")
news["headline"] = [clean_headline(t, s) for t, s in zip(news["title"], news["source"].fillna(""))]

top_all = []
for person in watch:
    top = (news[news["person"] == person]
           .sort_values("date", ascending=False)
           .drop_duplicates("headline")
           .head(5)
           .copy())
    top["한글 번역"] = [to_korean(h) for h in top["headline"]]
    top_all.append(top)
    print(f"\n■ {person} — 최신 기사 {len(top)}건")
    display(top[["date", "source", "headline", "한글 번역"]]
            .rename(columns={"date": "날짜", "source": "언론사", "headline": "원문 헤드라인"})
            .reset_index(drop=True))

if top_all:
    result = pd.concat(top_all)
    result.to_csv(f"{PREFIX}_news_top5_ko.csv", index=False, encoding="utf-8-sig")
    ok = (~result["한글 번역"].str.startswith("(")).sum()
    print(f"\n✅ 번역본 저장: {PREFIX}_news_top5_ko.csv · 번역 성공 {ok}/{len(result)}건 (엔진: {engine_name})")
    print("   기계 번역이라 인명·정치 용어는 원문과 함께 확인하세요.")


# ───────────────────────────────────────────────
# 9-2단계: 셀레니움으로 기사 원문 페이지 확인
#   - RSS는 기사 "목록"의 공식 창구, 셀레니움은 각 기사 "페이지"를 여는 데만 사용
#   - 구글 뉴스 링크는 자바스크립트 리다이렉트라 requests로는 언론사에 도달하지 못함
#   - 도착한 언론사의 robots.txt가 허용할 때만, 언론사가 공개한 요약·제목·발행일만 저장
# ───────────────────────────────────────────────
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

news["lead"] = ""
lead_rows = []
if USE_SELENIUM and top_all:
    try:
        try:
            from selenium import webdriver
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "selenium"])
            importlib.invalidate_caches()
            from selenium import webdriver
        from selenium.webdriver.support.ui import WebDriverWait

        opts = webdriver.ChromeOptions()
        if SELENIUM_HEADLESS:
            opts.add_argument("--headless=new")
        opts.add_argument("--lang=fr-FR")
        opts.add_argument(f"--user-agent={HEADERS['User-Agent']}")
        driver = webdriver.Chrome(options=opts)        # Selenium 4.6+는 드라이버 자동 설치
        driver.set_page_load_timeout(25)
        print("\n" + "═" * 60)
        print("9-2단계: 셀레니움으로 기사 원문 페이지 확인")
        print("═" * 60)

        robots_cache = {}

        def robots_ok(url):
            dom = urlparse(url).netloc
            if dom not in robots_cache:
                rp = RobotFileParser()
                rp.set_url(f"{urlparse(url).scheme}://{dom}/robots.txt")
                try:
                    rp.read()
                    robots_cache[dom] = rp
                except Exception:
                    robots_cache[dom] = None             # robots.txt를 못 읽으면 보수적으로 불허
            rp = robots_cache[dom]
            return bool(rp and rp.can_fetch(HEADERS["User-Agent"], url))

        META_JS = """
        const m = s => { const e = document.querySelector(s); return e ? (e.content || e.getAttribute('content') || '') : ''; };
        return {title: m('meta[property="og:title"]') || document.title,
                desc:  m('meta[property="og:description"]') || m('meta[name="description"]'),
                pub:   m('meta[property="article:published_time"]') || m('meta[name="date"]')};
        """
        try:
            for top in top_all:
                for _, a in top.head(SELENIUM_MAX_PER_PERSON).iterrows():
                    rec = {"person": a["person"], "url": a["url"], "headline": a["headline"],
                           "final_url": "", "status": "", "og_title": "", "lead": "", "published_meta": ""}
                    try:
                        driver.get(a["url"])
                        WebDriverWait(driver, 15).until(lambda d: "news.google.com" not in d.current_url)
                        final = driver.current_url
                        rec["final_url"] = final
                        if "consent." in final:
                            rec["status"] = "동의 페이지에서 멈춤"
                        elif not robots_ok(final):
                            rec["status"] = "robots.txt 불허 → 미수집"
                        else:
                            meta = driver.execute_script(META_JS) or {}
                            rec.update(status="수집", og_title=(meta.get("title") or "")[:200],
                                       lead=(meta.get("desc") or "")[:300],
                                       published_meta=(meta.get("pub") or "")[:25])
                    except Exception as e:
                        rec["status"] = f"실패 ({type(e).__name__})"
                    lead_rows.append(rec)
                    print(f"  [{rec['status']}] {a['person']} | {a['headline'][:60]}")
                    if rec["lead"]:
                        print(f"      요약: {rec['lead'][:120]}")
                    time.sleep(3)                          # 언론사별 요청 간격
        finally:
            driver.quit()
    except Exception as e:
        print(f"\n⚠ 셀레니움 사용 불가 ({type(e).__name__}: {str(e)[:80]}) → 헤드라인만으로 진행")

if lead_rows:
    leads = pd.DataFrame(lead_rows)
    leads.to_csv(f"{PREFIX}_news_leads.csv", index=False, encoding="utf-8-sig")
    lead_map = dict(zip(leads["url"], leads["lead"].fillna("")))
    news["lead"] = news["url"].map(lead_map).fillna("")
    ok = (leads["status"] == "수집").sum()
    print(f"\n✅ 기사 페이지 확인 {len(leads)}건 · 요약 수집 {ok}건 · 저장: {PREFIX}_news_leads.csv")
    print("   ※ 언론사가 공개한 요약(meta description)만 저장하며 본문 전체는 수집하지 않아요.")


# ───────────────────────────────────────────────
# 10단계: 전체 기사 분류 (행위 유형·인물 역할) + 최근 5건 한줄요약
# ───────────────────────────────────────────────
import json

ACTIONS = [
    ("무장해제·평화",  r"deposer les armes|desarm|reddition|\bddr\b"),
    ("협상·중재",      r"negoci|mediat|dialogue|paix|accord|pacification|reconcili"),
    ("억류·피해",      r"otage|libere|enlev|kidnap|detenu|sequestr"),
    ("무장 활동",      r"attaque|milice|rebelle|massacre|combat|insurg|groupe arme"),   # "armee/armes" 오탐 방지
    ("혐오·선동",      r"haine|incitation|tribal|xenophob|discours de haine"),
    ("사법·제재",      r"sanction|tribunal|\bcpi\b|proces|condamn|arrest|poursuite|inculp|enquete"),
    ("경제·자원",      r"\bmine|minerai|\bor\b|coltan|contrat|fonds|budget|corruption|detournement|blanchi"),
    ("정치 활동",      r"depute|election|parti|assemblee|nomination|ministre|candidat|gouvern|senat"),
    ("발언·입장 표명", r"declar|propos|discours|denonce|appel|plaide"),
]
ROLES = ["행위 주체", "발언 주체", "언급 대상", "피해자"]


def _norm(s):
    return strip_accents(str(s)).lower()


def classify(headline):
    """헤드라인 → (행위 유형 목록, 인물 역할)"""
    t = _norm(headline)
    hits = [name for name, pat in ACTIONS if re.search(pat, t)]
    if re.search(r"otage|libere|victime|enlev|detenu|sequestr", t):
        role = "피해자"
    elif re.search(r"accus|inculp|condamn|soupcon|implique|mis en cause|poursuivi", t):
        role = "행위 주체"
    elif re.search(r":\s*[«\"']", str(headline)):
        role = "발언 주체"
    else:
        role = "언급 대상"
    return hits, role


LLM_PROMPT = """너는 금융기관 KYC 담당 분석가다. 아래는 제재 대상 PEP에 관한 뉴스 헤드라인이다.
인물: {person}
헤드라인(프랑스어): {headline}
날짜: {date}

헤드라인에 있는 정보만 근거로 판단하고, 없는 내용은 추측하지 마라.
아래 JSON 형식으로만 답하라. 다른 문장은 쓰지 마라.
{{"행위유형": "{actions} 중 하나 또는 기타 언급",
 "역할": "{roles} 중 하나",
 "한줄요약": "정치경제적 의미를 담은 한국어 한 문장",
 "근거": "판단 근거 한 문장"}}"""

llm_client = None
if USE_LLM:
    try:
        try:
            from anthropic import Anthropic
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])
            importlib.invalidate_caches()
            from anthropic import Anthropic
        if not os.environ.get("ANTHROPIC_API_KEY"):
            raise RuntimeError("ANTHROPIC_API_KEY 환경변수가 없어요")
        llm_client = Anthropic()
        print(f"\n10단계: 전체 기사는 키워드 분류, 최근 5건은 Claude API({LLM_MODEL})로 문맥 판단")
    except Exception as e:
        print(f"\n⚠ Claude API 사용 불가 ({e}) → 키워드 규칙만 사용")
if llm_client is None:
    print("\n10단계: 키워드 규칙으로 분류 (문맥 판단은 사람이 검토 필요)")


def llm_classify(person, headline, date):
    msg = llm_client.messages.create(
        model=LLM_MODEL, max_tokens=400,
        messages=[{"role": "user", "content": LLM_PROMPT.format(
            person=person, headline=headline, date=date,
            actions=", ".join(a for a, _ in ACTIONS), roles=", ".join(ROLES))}])
    text = "".join(b.text for b in msg.content if b.type == "text")
    return json.loads(re.sub(r"```json|```", "", text).strip())


# 10-1) 전체 기사 키워드 분류 (9단계의 news 사용)
news_cls = news.dropna(subset=["headline"]).drop_duplicates(["person", "url"]).copy()
# 9-2단계에서 기사 요약을 얻었으면 헤드라인 + 요약으로 분류 (역할·행위를 더 정확히 판단)
cls = [classify(f"{h} {l}" if l else h) for h, l in zip(news_cls["headline"], news_cls["lead"].fillna(""))]
news_cls["분류 근거"] = ["키워드 규칙 (헤드라인+요약)" if l else "키워드 규칙" for l in news_cls["lead"].fillna("")]
news_cls["행위유형"] = [h for h, _ in cls]
news_cls["역할"] = [r for _, r in cls]

# 10-2) 최근 5건 한줄요약 (LLM이 있으면 그 판단으로 분류도 덮어씀)
top_urls = set(pd.concat(top_all)["url"]) if top_all else set()
sum_rows = []
for idx, r in news_cls[news_cls["url"].isin(top_urls)].iterrows():
    ko = to_korean(r["headline"]) if translator else ""
    summary_line = ko or r["headline"]
    if llm_client is not None:
        try:
            res = llm_classify(r["person"], r["headline"], r["date"])
            act = res.get("행위유형", "")
            news_cls.at[idx, "행위유형"] = [act] if act in dict(ACTIONS) else []
            news_cls.at[idx, "역할"] = res.get("역할", r["역할"]) if res.get("역할") in ROLES else r["역할"]
            news_cls.at[idx, "분류 근거"] = "Claude: " + res.get("근거", "")
            summary_line = res.get("한줄요약", ko)
        except Exception as e:
            news_cls.at[idx, "분류 근거"] = f"API 오류로 키워드 유지 ({type(e).__name__})"
        time.sleep(1)
    sum_rows.append({"인물": r["person"], "날짜": r["date"], "원문 헤드라인": r["headline"],
                     "행위유형": " / ".join(news_cls.at[idx, "행위유형"]) or "기타 언급",
                     "역할": news_cls.at[idx, "역할"], "한줄요약": summary_line})

news_summary = pd.DataFrame(sum_rows)
news_summary.to_csv(f"{PREFIX}_news_summary.csv", index=False, encoding="utf-8-sig")
for person, g in news_summary.groupby("인물", sort=False):
    print(f"\n■ {person} — 최근 기사 {len(g)}건 한줄요약")
    for _, r in g.sort_values("날짜", ascending=False).iterrows():
        print(f"  [{r['날짜']}] {r['행위유형']} | 역할: {r['역할']}")
        print(f"      → {r['한줄요약']}")
print(f"\n✅ 한줄요약 저장: {PREFIX}_news_summary.csv (전체 {len(news_cls)}건 분류 완료)")


# ───────────────────────────────────────────────
# 11단계: 기사 기반 위험 가중치 계산
#
# 근거 (설명용으로 단순화한 적용이며, 원문 확인 필요)
#  - FATF 위험기반접근법: 위험에 비례한 통제, 법적 지위가 기본값을 정함
#  - Wolfsberg 부정적 언론 가이드의 관점: 관련성(동일인)·신뢰성(출처)·중대성(혐의)
#  - 다기준 의사결정: A급은 비보상형(오버라이드), B·C급만 보상형 조정
#  - AHP(Saaty): 제재 카테고리 기본 치명도 산출, 일관성 비율(CR) 검증
#  - 증거 결합: 독립 사건만 세고(중복 보도 묶음), noisy-OR로 체감 결합
#  - 시간 감쇠: 지수 감쇠(반감기), 출처 신뢰도: Admiralty식 등급
# ───────────────────────────────────────────────
import numpy as np

# 11-1) AHP로 카테고리 기본 치명도 계산
AHP_DIMS = ["법적 거래금지·동결", "제재금·처벌", "환거래·달러 접근", "평판"]
AHP_MATRIX = np.array([[1,   2,   3,   5],
                       [1/2, 1,   2,   4],
                       [1/3, 1/2, 1,   3],
                       [1/5, 1/4, 1/3, 1]], dtype=float)
CAT_PROFILE = {"A": [1.0, 1.0, 1.0, 1.0],
               "B": [0.3, 0.5, 0.6, 0.7],
               "C": [0.0, 0.1, 0.2, 0.5]}

_vals, _vecs = np.linalg.eig(AHP_MATRIX)
_i = int(np.argmax(_vals.real))
AHP_W = _vecs[:, _i].real / _vecs[:, _i].real.sum()
_n = len(AHP_DIMS)
AHP_CR = ((_vals[_i].real - _n) / (_n - 1)) / 0.90          # 무작위지수 RI(n=4)=0.90
BASE_SEV = {k: float(np.dot(AHP_W, v)) for k, v in CAT_PROFILE.items()}

print("\n" + "═" * 60)
print("11단계: 기사 기반 위험 가중치")
print("═" * 60)
print("AHP 가중치:", ", ".join(f"{d} {w:.2f}" for d, w in zip(AHP_DIMS, AHP_W)),
      f"| CR = {AHP_CR:.3f} ({'일관성 충족' if AHP_CR < 0.1 else '재검토 필요'})")
print("카테고리 기본 치명도:", ", ".join(f"{k} {v:.2f}" for k, v in BASE_SEV.items()))

# 11-2) 매개변수 (모두 설명용 가정값 — 기관 정책에 맞게 조정)
HALF_LIFE_YEARS = 3.0                        # 시간 감쇠 반감기
CAP_UP, CAP_DOWN = 0.15, 0.05                # 기사에 의한 최대 가산 / 차감
FLOOR = {"B": 0.30, "C": 0.05}               # 제재 유효 기간 중 하한
ID_CONF = {"확정": 1.0, "동일인 후보": 0.8}  # 동일인 확신도
TYPE_EFFECT = {                              # (방향, 중대성)
    "무장 활동": (+1, 1.0), "혐오·선동": (+1, 0.9), "사법·제재": (+1, 0.8), "경제·자원": (+1, 0.6),
    "무장해제·평화": (-1, 0.6), "협상·중재": (-1, 0.5),
}
ROLE_W_UP = {"행위 주체": 1.0, "언급 대상": 0.6, "발언 주체": 0.4, "피해자": 0.0}
ROLE_W_DOWN = {"발언 주체": 1.0, "행위 주체": 1.0, "언급 대상": 0.6, "피해자": 0.0}
STAGE = [(r"condamn", 1.0), (r"inculp|poursui|arrest|mandat d.arret", 0.8),
         (r"enquete|accus|soupcon|allegation|mis en cause", 0.6)]
STAGE_DEFAULT = 0.7
SOURCE_TIER = [  # (패턴, 신뢰도) — Admiralty식 출처 등급을 단순화
    (r"reuters|afp|associated press|\bap\b|rfi|bbc|le monde|radio okapi|jeune afrique|un news|onu", 1.0),
    (r"actualite\.cd|7sur7|mediacongo|politico\.cd|top congo|radio france|france 24|tv5", 0.8),
]
SOURCE_DEFAULT = 0.6
DELIST = r"levee? des sanctions|radie|retire de la liste|fin des sanctions|annul.*sanction|delist"
NEW_DESIG = r"nouvelles? sanctions|sanctionne par|ajoute a la liste|ofac|tresor americain|conseil de securite.*sanction"
REVIEW_FILE = f"{PREFIX}_news_review.csv"     # 검토자 수정 파일 (있으면 우선 적용)


def source_cred(src):
    t = _norm(src)
    for pat, w in SOURCE_TIER:
        if re.search(pat, t):
            return w
    return SOURCE_DEFAULT


def stage_of(headline):
    t = _norm(headline)
    for pat, w in STAGE:
        if re.search(pat, t):
            return w
    return STAGE_DEFAULT


def decay(date_str):
    d = pd.to_datetime(date_str, errors="coerce", utc=True)
    if pd.isna(d):
        return 0.5                                    # 날짜 불명 → 반감기 1회 적용
    age_y = (pd.Timestamp.now(tz="UTC") - d).days / 365.25
    return float(0.5 ** (max(age_y, 0) / HALF_LIFE_YEARS))


# 11-3) 검토자 수정 반영 (url 기준: 검토_행위유형, 검토_역할, 검토_제외=Y)
if os.path.exists(REVIEW_FILE):
    rev = pd.read_csv(REVIEW_FILE).set_index("url")
    applied = 0
    for idx, r in news_cls.iterrows():
        if r["url"] in rev.index:
            rv = rev.loc[r["url"]]
            if str(rv.get("검토_제외", "")).upper() == "Y":
                news_cls.at[idx, "행위유형"], news_cls.at[idx, "역할"] = [], "피해자"
                news_cls.at[idx, "분류 근거"] = "검토자: 제외"
            else:
                if pd.notna(rv.get("검토_행위유형")) and str(rv["검토_행위유형"]).strip():
                    news_cls.at[idx, "행위유형"] = [x.strip() for x in str(rv["검토_행위유형"]).split("/")]
                if pd.notna(rv.get("검토_역할")) and rv["검토_역할"] in ROLES:
                    news_cls.at[idx, "역할"] = rv["검토_역할"]
                news_cls.at[idx, "분류 근거"] = "검토자 수정"
            applied += 1
    print(f"검토자 수정 {applied}건 반영 ({REVIEW_FILE})")

# 11-4) 기사별 점수 = 동일인 확신도 × 중대성 × 절차 단계 × 출처 신뢰도 × 시간 감쇠 × 역할
verdict_of = {t[0]: t[1] for t in targets}
cat_of = dict(zip(cat_df["인물"], cat_df["최종 카테고리"])) if len(cat_rows) else {}

art_rows = []
for _, r in news_cls.iterrows():
    person, hl = r["person"], r["headline"]
    idc = ID_CONF.get(verdict_of.get(person, "동일인 후보"), 0.8)
    src, stg, dec = source_cred(r.get("source", "")), stage_of(hl), decay(r["date"])
    up = max([TYPE_EFFECT[h][1] for h in r["행위유형"] if TYPE_EFFECT.get(h, (0,))[0] > 0], default=0)
    down = max([TYPE_EFFECT[h][1] for h in r["행위유형"] if TYPE_EFFECT.get(h, (0,))[0] < 0], default=0)
    s_up = idc * up * stg * src * dec * ROLE_W_UP.get(r["역할"], 0.6)
    s_down = idc * down * src * dec * ROLE_W_DOWN.get(r["역할"], 0.6)
    t = _norm(hl)
    art_rows.append({"person": person, "date": r["date"], "source": r.get("source", ""),
                     "headline": hl, "url": r["url"],
                     "행위유형": " / ".join(r["행위유형"]) or "기타 언급", "역할": r["역할"],
                     "동일인": idc, "출처": src, "단계": stg, "감쇠": round(dec, 2),
                     "가산 점수": round(s_up, 3), "차감 점수": round(s_down, 3),
                     "해제 보도": bool(re.search(DELIST, t)), "추가 지정 보도": bool(re.search(NEW_DESIG, t)),
                     "분류 근거": r["분류 근거"]})
arts = pd.DataFrame(art_rows)

# 11-5) 같은 사건 묶기: 같은 인물, 날짜 3일 이내, 헤드라인 단어 유사도(Jaccard) 0.4 이상
STOP_FR = set("le la les de des du un une et en au aux pour par sur dans avec est sont qui que rdc congo".split())

def _toks(h, person):
    name = set(tokens(person))
    return {w for w in re.findall(r"[a-z]{4,}", _norm(h)) if w not in STOP_FR and w not in name}

arts["_dt"] = pd.to_datetime(arts["date"], errors="coerce", utc=True)
arts["사건ID"] = -1
cid = 0
for person, g in arts.sort_values("_dt").groupby("person"):
    reps = []                                        # (사건ID, 날짜, 단어집합)
    for idx, r in g.iterrows():
        tk, assigned = _toks(r["headline"], person), None
        for c, dt, rtk in reps:
            close = pd.isna(dt) or pd.isna(r["_dt"]) or abs((r["_dt"] - dt).days) <= 3
            jac = len(tk & rtk) / max(len(tk | rtk), 1)
            if close and jac >= 0.4:
                assigned = c
                break
        if assigned is None:
            assigned, cid = cid, cid + 1
            reps.append((assigned, r["_dt"], tk))
        arts.at[idx, "사건ID"] = assigned

# 11-6) 인물별 결합: 사건별 최고 점수 → noisy-OR → 상한·하한 적용
def noisy_or(scores):
    return 1 - float(np.prod([1 - min(max(s, 0), 1) for s in scores])) if len(scores) else 0.0

w_rows = []
for person in watch:
    g = arts[arts["person"] == person]
    cat = cat_of.get(person, "C")
    base = BASE_SEV[cat]
    ev = g.groupby("사건ID").agg(up=("가산 점수", "max"), down=("차감 점수", "max"))
    p_up, p_down = noisy_or(ev["up"].tolist()), noisy_or(ev["down"].tolist())
    flags = []
    if g["해제 보도"].any():
        flags.append("해제 보도 감지 → 공식 공시 확인 후 재분류")
    if g["추가 지정 보도"].any():
        flags.append("추가 지정 보도 감지 → A급 전환 검토")
    id_support = int(g["행위유형"].str.contains("무장|평화|중재|억류").sum())

    if cat == "A":
        adj, final = 0.0, 1.0
        note = "오버라이드: 기사로 조정하지 않음 (거래 금지 유지)"
        if verdict_of.get(person) == "동일인 후보" and id_support:
            note += f" · 동일인 확인 근거 기사 {id_support}건 → 동일인 확정 검토"
    else:
        adj = CAP_UP * p_up - CAP_DOWN * p_down
        final = min(max(base + adj, FLOOR.get(cat, 0.0)), 1.0)
        note = "가산 우세" if adj > 0.005 else "차감 우세" if adj < -0.005 else "조정 없음"

    w_rows.append({"인물": person, "판정": verdict_of.get(person, ""), "카테고리": cat,
                   "기본 치명도": round(base, 3), "기사 수": len(g), "독립 사건 수": len(ev),
                   "가산 증거(P+)": round(p_up, 3), "차감 증거(P-)": round(p_down, 3),
                   "조정": round(adj, 3), "최종 가중치": round(final, 3),
                   "해석": note, "재분류 신호": " / ".join(flags) or "없음"})

weights = pd.DataFrame(w_rows)
print("\n[인물별 최종 가중치]")
display(weights.reset_index(drop=True))

for person in watch:
    g = arts[arts["person"] == person].copy()
    g["기여"] = g[["가산 점수", "차감 점수"]].max(axis=1)
    g = g[g["기여"] > 0].sort_values("기여", ascending=False).head(5)
    if len(g):
        print(f"\n■ {person} — 가중치에 기여한 기사 상위 {len(g)}건")
        display(g[["date", "source", "headline", "행위유형", "역할", "출처", "단계", "감쇠",
                   "가산 점수", "차감 점수"]].rename(columns={"date": "날짜", "source": "언론사",
                                                          "headline": "원문 헤드라인"}).reset_index(drop=True))
    else:
        print(f"\n■ {person} — 가중치에 기여한 기사 없음 (일반 정치 활동·피해자·기타 보도)")

arts.drop(columns=["_dt"]).to_csv(f"{PREFIX}_article_scores.csv", index=False, encoding="utf-8-sig")
weights.to_csv(f"{PREFIX}_weight_adjustment.csv", index=False, encoding="utf-8-sig")
if not os.path.exists(REVIEW_FILE):
    tmpl = arts[["person", "date", "headline", "url", "행위유형", "역할"]].copy()
    tmpl["검토_행위유형"], tmpl["검토_역할"], tmpl["검토_제외"] = "", "", ""
    tmpl.to_csv(f"{PREFIX}_news_review_template.csv", index=False, encoding="utf-8-sig")
    print(f"\n검토 양식 저장: {PREFIX}_news_review_template.csv → 수정 후 {REVIEW_FILE}로 저장하면 다음 실행에 반영돼요.")
print(f"✅ 저장: {PREFIX}_article_scores.csv, {PREFIX}_weight_adjustment.csv")
print("   ※ 모든 매개변수는 설명용 가정값이며, 최종 판단은 검토자가 기사 본문과 공식 공시로 확인해야 해요.")


# ═══════════════════════════════════════════════
# 파이프라인 완료 요약
# ═══════════════════════════════════════════════
print("\n" + "═" * 60)
print("파이프라인 완료 요약")
print("═" * 60)
print(f"[수집] {COUNTRY} 의원 {len(peps)}명 · 제재명단 {len(sanc):,}건")
print(f"[대조] 확정 {len(id_hits)}명 · 동일인 후보 {int(cands['birth_match'].sum())}명 · "
      f"오탐 {int((~cands['birth_match']).sum())}명 → 감시 대상 {len(watch)}명")
print(f"[뉴스] 수집 기사 {len(news)}건 · 분류 {len(news_cls)}건")

# 5단계 스크리닝 요약에 누적 기사 수를 붙여 다시 저장
if len(summary):
    summary["누적 기사 수"] = summary["인물"].map(news["person"].value_counts()).fillna(0).astype(int)
    summary.to_csv(f"{PREFIX}_summary.csv", index=False, encoding="utf-8-sig")
    print(summary[["인물", "판정", "제재 목록 수", "누적 기사 수"]].to_string(index=False))
for _, r in weights.iterrows():
    print(f"[위험] {r['인물']}: {r['카테고리']}급 · 기본 {r['기본 치명도']:.2f} → 최종 {r['최종 가중치']:.2f} "
          f"({r['해석']}) · 재분류 신호: {r['재분류 신호']}")
print("\n생성 파일:")
for f in ["peps.csv", "person_profile.csv", "sanction_id_hits.csv", "sanction_candidates.csv", "summary.csv",
          "sanction_labeled.csv", "sanction_category.csv", "sanction_news_monitor.csv",
          "news_top5_ko.csv", "news_leads.csv", "news_summary.csv", "article_scores.csv", "weight_adjustment.csv"]:
    path = f"{PREFIX}_{f}"
    print(f"  {'✅' if os.path.exists(path) else '—'} {path}")
print("\n※ 모든 기준·가중치는 설명용 가정값이에요. CSV는 .gitignore로 저장소에서 제외하세요.")

상태 코드: 200

DR Congo 데이터셋 1개
 - DR Congo Members of the National Assembly (cd_national_assembly)
✅ DR Congo Members of the National Assembly: 560행

인물 560명 (예상 약 561명)
제재명단 73,810건

[제재 ID 일치 — 확정] 1명
                        name birth_date  sanctions                                                                                                                                                                                                                                       dataset_sanc
Bitakwira Bihona Hayi Justin 1960-12-05        NaN Belgian Financial Sanctions;EU Consolidated Travel Bans;EU Council Official Journal Sanctioned Entities;EU Financial Sanctions Files (FSF);French National Asset Freezing System;Monaco National Fund Freezing List;Swiss SECO Sanctions/Embargoes

[이름·별칭 일치 — 후보] 2건 (생년월일 일치 1건)
                        pep_name     matched_name              verdict
          Ndjabu Ngabu Floribert Floribert Ndjabu 동일인 가능성 높음 (생년월일 일치)
Wagner Tshienda Tshienda Richard   R

,인물,데이터셋,기관,프로그램,지정 사유,지정일
0,Bitakwira Bihona Hayi Justin,eu_fsf,"Directorate‑General for Financial Stability, Financial Services and Capital Markets Union",COD,2023/2771 (OJ L11122013),2023-12-12
1,Ndjabu Ngabu Floribert,un_sc_sanctions,United Nations Security Council,DRC,,2005-11-01
2,Ndjabu Ngabu Floribert,eu_fsf,"Directorate‑General for Financial Stability, Financial Services and Capital Markets Union",COD,2017/396 (OJ L60),2017-03-08
3,Ndjabu Ngabu Floribert,gb_fcdo_sanctions,"UN; Foreign, Commonwealth & Development Office",The Democratic Republic of the Congo (Sanctions) (EU Exit) Regulations 2019,,2005-11-01



✅ 저장: drc_person_profile.csv  (공식 데이터에 있는 항목만 표시, 없는 항목은 '정보 없음')

════════════════════════════════════════════════════════════
요약: DR Congo 의원 560명 중
  · 제재 확정 1명
  · 동일인 가능성 높은 후보 1명
  · 오탐으로 판단한 후보 1명
                              인물                   판정  제재 목록 수
    Bitakwira Bihona Hayi Justin           확정 (ID 일치)        7
          Ndjabu Ngabu Floribert 동일인 가능성 높음 (생년월일 일치)       15
Wagner Tshienda Tshienda Richard 오탐 가능성 (생년월일 불일치·없음)        1
════════════════════════════════════════════════════════════

✅ 스크리닝 결과 저장 완료

════════════════════════════════════════════════════════════
제재 출처·명목·카테고리
기관 가정: EU 연결 False, 영국 연결 False, 달러 결제 True
════════════════════════════════════════════════════════════


,인물,제재 목록,원지정 기관,구분,조치,카테고리
0,Bitakwira Bihona Hayi Justin,Belgian Financial Sanctions,EU,이행(벨기에),자산동결,B
1,Bitakwira Bihona Hayi Justin,EU Financial Sanctions Files (FSF),EU,원지정,자산동결,B
2,Bitakwira Bihona Hayi Justin,French National Asset Freezing System,EU,이행(프랑스),자산동결,B
3,Bitakwira Bihona Hayi Justin,Monaco National Fund Freezing List,EU,이행(모나코),자산동결,B
4,Bitakwira Bihona Hayi Justin,Swiss SECO Sanctions/Embargoes,EU,이행(스위스),자산동결,B
5,Bitakwira Bihona Hayi Justin,EU Consolidated Travel Bans,EU,원지정,여행금지,C
6,Bitakwira Bihona Hayi Justin,EU Council Official Journal Sanctioned Entities,EU,원지정,법적 지정 근거(관보),C
7,Ndjabu Ngabu Floribert,Argentina RePET Sanctions,UN 안보리,이행(아르헨티나),자산동결,A
8,Ndjabu Ngabu Floribert,Australian Sanctions Consolidated List,UN 안보리,이행(호주),자산동결,A
9,Ndjabu Ngabu Floribert,Japan Economic Sanctions and List of Eligible People,UN 안보리,이행(일본),자산동결,A



[인물별 최종 카테고리]


...


■ Bitakwira Bihona Hayi Justin — 가중치에 기여한 기사 상위 5건


,날짜,언론사,원문 헤드라인,행위유형,역할,출처,단계,감쇠,가산 점수,차감 점수
0,2026-07-11,mediacongo.net,Actualité | Sud-Kivu : L’élu d’Uvira Justin Bitakwira tire la sonnette d’alarme sur l’avancée des rebelles vers Baraka,무장 활동,언급 대상,0.8,0.7,0.93,0.313,0.000
1,2023-07-21,Ouragan.cd,"Bitakwira, blâmes et condamnations malgré son audition au parquet",사법·제재,행위 주체,0.6,1.0,0.47,0.225,0.000
2,2026-06-08,mediacongo.net,Actualité | Justin Bitakwira plaide pour un dialogue entre majorité et opposition afin d’éviter une crise politique,협상·중재 / 발언·입장 표명,언급 대상,0.8,0.7,0.91,0.000,0.219
3,2023-07-23,Scoop RDC,"Accusé de raciste et haineux, Bitakwira à l’inspecteur du parquet près la Cour d’appel de Kinshasa-Gombe : « Je ne saurais répondre à différentes questions que vous allez me poser si vous n’avez pas lu le livre Holocauste au Congo de Charles Onana",혐오·선동 / 발언·입장 표명,행위 주체,0.6,0.6,0.47,0.152,0.000
4,2023-07-19,mediacongo.net,Actualité | L'UE attire l'attention du ministre de la justice sur Justin Bitakwira pour son discours de haine,혐오·선동 / 정치 활동 / 발언·입장 표명,언급 대상,0.8,0.7,0.47,0.142,0.000



■ Ndjabu Ngabu Floribert — 가중치에 기여한 기사 상위 3건


,날짜,언론사,원문 헤드라인,행위유형,역할,출처,단계,감쇠,가산 점수,차감 점수
0,2023-02-03,DW.com,"Floribert Ndjabu : ""Ils acceptent de déposer les armes""",무장해제·평화,발언 주체,0.6,0.7,0.42,0.00,0.122
1,2020-04-22,Radio Okapi |,"Libération de Floribert Ndajbu : son parti se réjouit, une ONG regrette l’absence d’un procès",사법·제재 / 정치 활동,언급 대상,1.0,0.7,0.22,0.06,0.000
2,2020-09-23,VOA Afrique,D'ex-chefs de guerre en émissaires de paix auprès des miliciens en Ituri,협상·중재,언급 대상,0.6,0.7,0.24,0.00,0.035



검토 양식 저장: drc_news_review_template.csv → 수정 후 drc_news_review.csv로 저장하면 다음 실행에 반영돼요.
✅ 저장: drc_article_scores.csv, drc_weight_adjustment.csv
   ※ 모든 매개변수는 설명용 가정값이며, 최종 판단은 검토자가 기사 본문과 공식 공시로 확인해야 해요.

════════════════════════════════════════════════════════════
파이프라인 완료 요약
════════════════════════════════════════════════════════════
[수집] DR Congo 의원 560명 · 제재명단 73,810건
[대조] 확정 1명 · 동일인 후보 1명 · 오탐 1명 → 감시 대상 2명
[뉴스] 수집 기사 84건 · 분류 84건
                              인물                   판정  제재 목록 수  누적 기사 수
    Bitakwira Bihona Hayi Justin           확정 (ID 일치)        7       75
          Ndjabu Ngabu Floribert 동일인 가능성 높음 (생년월일 일치)       15        9
Wagner Tshienda Tshienda Richard 오탐 가능성 (생년월일 불일치·없음)        1        0
[위험] Bitakwira Bihona Hayi Justin: B급 · 기본 0.44 → 최종 0.55 (가산 우세) · 재분류 신호: 해제 보도 감지 → 공식 공시 확인 후 재분류
[위험] Ndjabu Ngabu Floribert: A급 · 기본 1.00 → 최종 1.00 (오버라이드: 기사로 조정하지 않음 (거래 금지 유지) · 동일인 확인 근거 기사 4건 → 동일인 확정 검토) · 재분류 신호: 없음

생성 파일:
  ✅ drc_peps.csv
  ✅ drc_person_pro

# DR Congo 국회의원(PEP) 제재 스크리닝 EDA 평가서

## 요약

| 분석 질문 | 답 |
|---|---|
| Q1. DR Congo 국회의원 중 제재 대상 PEP가 있는가? | **있다. 560명 중 2명(0.36%)** — Bitakwira Bihona Hayi Justin(ID 일치 확정), Ndjabu Ngabu Floribert(이름·생년월일 일치, 동일인 가능성 높음). 이름만 겹친 1명은 오탐으로 제외 |
| Q2. 제재 정도는 인물별로 수치상 어느 정도인가? | **Ndjabu: A급(금지), 치명도 1.00** — UN 안보리 자산동결, 전 세계 15개 목록에 반영. **Bitakwira: B급(제한), 치명도 0.44** — EU 자산동결 중심, 한국 금융기관 기준 법적 금지는 아니나 EDD 대상 |
| Q3. 최근 뉴스는 제재 이행에 어떤 방향으로 작용하는가? | 구글 뉴스 RSS로 기사 목록을 받고, **셀레니움으로 각 기사 페이지를 열어 언론사가 공개한 요약까지 확인**해 판단했다. **Ndjabu: 위험을 높이는 기사 없음, 가중치 1.00 유지(오버라이드)** — 기사 속 역할은 무장 해제 중재자·억류 피해자이며, 기사는 위험 조정이 아니라 동일인 확인 근거로 작용. **Bitakwira: 기본 0.44에서 기사 내용에 따라 −0.05~+0.15 범위로 조정**(확정값은 `drc_weight_adjustment.csv`) — 74건이라는 기사 수 자체는 반영하지 않음 |

**핵심 메시지.** 같은 "제재 대상 PEP"라도 위험의 크기는 목록 수가 아니라 **원지정 기관(UN이냐 EU냐)과 기관과의 연결고리**로 결정된다. 뉴스는 A급에서는 "누구인가"를 확인하는 도구이고, B급에서는 "얼마나 위험한가"를 조정하는 도구다.

| 인물 | 판정 | 카테고리 | 기본 치명도 | 기사 수 | 뉴스에 의한 조정 | 은행 대응 |
|---|---|---|---|---|---|---|
| Ndjabu Ngabu Floribert | 동일인 후보 | A | 1.00 | 9건 | 없음 (오버라이드) | 거래 금지·동결·보고, 동일인 확정 검토 |
| Bitakwira Bihona Hayi Justin | 확정 | B | 0.44 | 74건 | −0.05 ~ +0.15 (실행 결과로 확정) | EDD·상급자 승인·고위험 하한 0.30 |

### 제재 대상 프로필

| 항목 | Ndjabu Ngabu Floribert | Bitakwira Bihona Hayi Justin |
|---|---|---|
| 현 직위 | DR Congo 국회의원 (`cd_national_assembly` 수록) | DR Congo 국회의원 (`cd_national_assembly` 수록) |
| 생년월일·나이 | 제재명단 생년월일과 일치 (값은 3-2단계 출력) | 1960-12-05 (만 65세, 2026년 10월 기준) |
| 동일인 판정 | 동일인 가능성 높음 (이름 + 생년월일) | 확정 (OpenSanctions ID 일치) |
| 제재 원지정 기관 | UN 안보리 · EU · 영국 | EU |
| 가장 강한 조치 | UN 자산동결·여행금지 | EU 자산동결 |
| 제재 목록 수 | 15개 (이행 포함) | 7개 (이행 포함) |
| 등급·치명도 | A급 · 1.00 | B급 · 0.44 |
| 언론 노출 | 9건 | 74건 |
| 기사 속 역할 | 무장 해제 중재자·억류 피해자, 이투리 무장단체 출신 인물(토마 루방가)과 동시 출현 | 행위 유형·역할 분류 결과로 판단 (11단계) |
| 은행 대응 | 거래 금지·동결·보고, 동일인 확정 검토 최우선 | EDD·상급자 승인, 기사 동향으로 위험 조정 |

두 사람은 모두 **현직 국회의원(PEP)이면서 국제 제재 대상**이라는 공통점이 있지만, 제재의 근거와 범위가 다르다. Ndjabu는 UN 안보리 지정을 기반으로 전 세계에 이행된 제재를, Bitakwira는 EU와 그 회원국 중심의 지역 제재를 받고 있다. 성별·출생지·국적·재직 기간·제재 기록상 직함·지정 사유 등 상세 인적사항은 파이프라인 3-2단계에서 OpenSanctions 상세 데이터(FtM)로 추출해 `drc_person_profile.csv`에 저장하며, 공식 데이터에 없는 항목은 "정보 없음"으로 둔다.

---

## 1. 분석 질문과 배경

본 분석은 금융기관 KYC 절차의 **제재 스크리닝**과 **부정적 언론 검색(adverse media screening)** 을 DR Congo 국회의원을 대상으로 축소 재현한 것이다.

| 질문 | KYC 단계 | 산출물 |
|---|---|---|
| Q1. 제재 대상 PEP가 있는가 | 명단 스크리닝, 오탐 해소 | 확정·후보·오탐 판정 |
| Q2. 제재 정도는 수치상 어느 정도인가 | 위험 등급화 | A/B/C 카테고리, AHP 치명도 |
| Q3. 최근 뉴스가 제재 이행에 어떤 방향으로 작용하는가 | 부정적 언론 검토, 지속 모니터링 | 기사별 점수, 인물별 가중치 조정 |

DR Congo를 대상으로 삼은 이유는 두 가지다. 국회의원이 560명으로 스크리닝 표본이 충분하고, UN 안보리 제재 체제가 장기간 운영되어 온 국가여서 PEP와 제재 대상이 겹치는 사례를 관찰하기에 적합하다. 분쟁 이후 무장세력 지도자가 정치권에 편입되는 과정에서 PEP와 제재 대상이 동시에 성립할 수 있다는 점도 분석 가치가 크다.

## 2. 데이터

| 데이터 | 출처 | 규모 | 접근 방식 |
|---|---|---|---|
| PEP 명단 | OpenSanctions `cd_national_assembly` (DR Congo 국회의원) | 560명 | `/pep/` 페이지 BeautifulSoup 파싱 → 공식 CSV |
| 제재명단 | OpenSanctions 제재 통합 파일 | 73,810건 | 공식 CSV (별칭·생년월일·제재 목록 포함) |
| 인물 상세 | OpenSanctions FtM (`cd_national_assembly`, `un_sc_sanctions`, `eu_fsf`, `gb_fcdo_sanctions`) | 감시 대상 2명 관련 기록 | 공식 상세 파일 다운로드 후 ID·이름·생년월일로 추출 |
| 뉴스 목록 | 구글 뉴스 RSS (프랑스어·콩고판) | 83건 (74 + 9) | 제재 인물 2명만 이름 검색 |
| 기사 요약 | 각 언론사 기사 페이지의 공개 요약(meta description)·제목·발행일 | 인물별 최신 5건 | 셀레니움(헤드리스 크롬), 언론사 robots.txt 허용 시만 |
| 번역 | deep-translator (구글 → MyMemory 자동 전환) | 인물별 최신 5건 | 기계 번역 |

도구는 페이지 유형에 따라 골랐다. OpenSanctions `/pep/` 페이지는 HTML 소스에 데이터가 있는 정적 페이지라 `requests` + BeautifulSoup으로 충분했고, 뉴스 목록은 자동 수신용 RSS라 `requests` + feedparser로 받았다. 반면 구글 뉴스의 기사 링크는 **자바스크립트로 언론사 페이지로 이동**시키는 방식이라 `requests`로는 언론사에 도달할 수 없어, 이 구간에만 **셀레니움**을 사용했다. robots.txt가 금지한 `/search`, `/entities`는 공식 다운로드 파일로 대체했다. 표의 엔티티 수(561)와 수집 인원(560)의 1건 차이는 인물이 아닌 항목이나 업데이트 시점 차이로 보이며 영향은 미미하다.

## 3. 조사 방법

### 3.1 파이프라인 구조

| 구간 | 단계 | 내용 | 주요 산출 파일 |
|---|---|---|---|
| 수집 | 1 | DR Congo 데이터셋 탐색 → 의원 명단 | `drc_peps.csv` |
| 대조 | 2–3 | 제재명단 대조, 감시 대상 선정 | `drc_sanction_id_hits.csv`, `drc_sanction_candidates.csv` |
| 프로필 | 3-2 | 감시 대상의 직위·나이·인적사항·제재 상세 추출 | `drc_person_profile.csv` |
| 제재 | 4–6 | 결과 저장, 요약, 목록 라벨링·A/B/C 카테고리 | `drc_summary.csv`, `drc_sanction_category.csv` |
| 뉴스 | 7–8 | RSS 검색 준비, 새 기사 수집 (30분 주기 자동 반복 가능) | `drc_sanction_news_monitor.csv` |
| 기사 | 9 | 최신 헤드라인 5건 번역 | `drc_news_top5_ko.csv` |
| 기사 | 9-2 | 셀레니움으로 기사 페이지 확인 → 언론사 공개 요약·발행일 | `drc_news_leads.csv` |
| 기사 | 10 | 전체 기사 행위 유형·역할 분류 (헤드라인 + 요약) | `drc_news_summary.csv` |
| 위험 | 11 | AHP 기본 치명도 + 기사 기반 가중치 조정 | `drc_weight_adjustment.csv`, `drc_article_scores.csv` |

전체가 `drc_sanction_monitor.py` 하나로 실행되며, "명단 → 제재 → 뉴스 → 기사 → 위험" 순으로 앞 단계의 결과가 다음 단계의 입력이 된다. 단계마다 개수를 출력해 결과가 비거나 틀린 채 진행되는 "조용한 실패"를 검증하도록 했다.

### 3.2 이름 매칭 (Q1)

| 단계 | 방법 | 판정 |
|---|---|---|
| 1 | OpenSanctions 공통 ID 일치 | 확정 |
| 2 | 제재명단 이름·별칭의 단어 묶음이 PEP 이름에 모두 포함 (어순·악센트 무관) | 후보 |
| 3 | 생년월일 비교 | 동일인 가능성 높음 / 오탐 |

DR Congo 공식 명단은 "성 + 후성(postnom) + 이름" 순서여서 제재명단이나 기사의 "이름 성" 표기와 어순이 다르다. 글자 그대로 비교하는 정확 일치로는 같은 사람을 놓치므로 2단계에서 어순과 무관하게 단어 묶음의 포함 여부로 후보를 찾았다. 이 방식은 흔한 이름끼리 우연히 겹칠 위험이 있어 3단계 생년월일 대조로 걸러내도록 설계했다.

### 3.3 제재 라벨링과 카테고리 (Q2)

각 제재 목록에 **원지정 기관**, **원지정·타국 이행 여부**, **조치 종류**를 라벨링한 뒤, 기관에 대한 구속력으로 카테고리를 나눴다. 각국 목록 상당수는 UN·EU 결정을 국내법으로 이행한 것이어서, 목록 수를 그대로 세면 같은 결정을 여러 번 세게 되기 때문이다. 기관은 "한국 금융기관, EU·영국 연결 없음, 달러 결제 있음"으로 가정했다.

| 카테고리 | 기준 | 처리 | 모델상 성격 |
|---|---|---|---|
| A 금지 | 구속력 있는 자산동결 (UN, 달러 결제 시 OFAC 등) | 거래 금지·동결·보고 | 비보상형 (오버라이드) |
| B 제한 | 구속력 없는 자산동결, 수출통제 | EDD·상급자 승인·고위험 하한 | 보상형 (조정 가능) |
| C 감시 | 여행금지, 감시목록, 지정 공시 | 모니터링 강화 | 보상형 (조정 가능) |

A급을 점수에 합산하지 않는 것은 다기준 의사결정의 **비보상형(사전식) 규칙**에 따른 것이다. 거래 금지 의무는 다른 위험 요인이 낮다고 상쇄될 수 없다.

### 3.4 치명도 정량화: AHP (Q2)

제재가 금융기관에 피해를 주는 네 경로를 Saaty 척도로 쌍대비교해 가중치를 산출했다.

| 비교 | 판단 | 근거 |
|---|---|---|
| 법적 금지 vs 처벌 | 2 | 금지 의무 위반이 처벌의 원인 |
| 법적 금지 vs 환거래 상실 | 3 | 환거래 상실은 상대 은행 판단에 따른 간접 영향 |
| 법적 금지 vs 평판 | 5 | 평판은 회복 가능·측정 불확실 |
| 처벌 vs 환거래 상실 | 2 | 제재금은 확정적 재무 손실 |
| 처벌 vs 평판 | 4 | 확정 손실 대 불확실 손실 |
| 환거래 상실 vs 평판 | 3 | 대리은행 단절은 해외 송금 업무 자체를 막음 |

| 영향 경로 | AHP 가중치 | A | B | C |
|---|---|---|---|---|
| 법적 거래금지·동결 | 0.47 | 1.0 | 0.3 | 0.0 |
| 제재금·처벌 | 0.28 | 1.0 | 0.5 | 0.1 |
| 환거래·달러 접근 | 0.17 | 1.0 | 0.6 | 0.2 |
| 평판 | 0.07 | 1.0 | 0.7 | 0.5 |
| **치명도 지수** | **CR = 0.019** | **1.00** | **0.44** | **0.10** |

일관성 비율(CR)이 0.1 미만으로 판단의 일관성은 충족했다. 경로 가중치는 AHP로, 카테고리별 경로 점수는 분석자가 직접 매겼다. B급이 구속력이 없는데도 A급의 절반 가까이 되는 것은, 해외 대리은행이 EU·영국 명단으로 스크리닝하기 때문에 **송금 차단·거래 관계 단절 위험**이 상당하기 때문이다.

### 3.5 뉴스 수집 (Q3)

GDELT API로 의원 전원을 검색하려 했으나 429(요청 과다)가 지수 백오프(30·60·120초) 후에도 해제되지 않았다. 제한을 우회하지 않고 설계를 바꿨다.

| 항목 | 초기 설계 | 최종 설계 |
|---|---|---|
| 검색 대상 | 의원 전원 (최대 560회 요청) | 제재 인물만 (2회 요청) |
| 소스 | GDELT API | 구글 뉴스 RSS (프랑스어·콩고판) |
| 검색어 | 공식 명단 이름 그대로 | "이름 성" / "성 이름" OR 검색 |
| 응답 처리 | 200 외 실패 | 302 리다이렉트 추적 |

**셀레니움을 쓴 구간과 이유.** 기사 목록은 공식 창구인 RSS로 받되, 헤드라인만으로는 "무슨 논란인지", "가해자인지 중재자인지"를 알기 어려운 경우가 많았다. 각 기사 페이지에는 언론사가 직접 공개한 요약(meta description)이 있지만, 구글 뉴스 링크는 자바스크립트로 이동하는 중간 주소라 일반 HTTP 요청으로는 열리지 않는다. 그래서 이 구간에만 실제 브라우저를 구동하는 셀레니움을 썼다.

| 구분 | RSS + requests | 셀레니움 (9-2단계) |
|---|---|---|
| 역할 | 기사 **목록** 수집 | 각 기사 **페이지** 확인 |
| 얻는 정보 | 헤드라인, 링크, 언론사, 날짜 | 언론사 공개 요약, 기사 제목, 발행일 |
| 필요한 이유 | 자동 수신용 형식이라 브라우저 불필요 | 자바스크립트 리다이렉트를 따라가야 언론사에 도달 |
| 대상 | 제재 인물 관련 기사 전체 | 인물별 최신 5건 |

**셀레니움 사용 원칙.** 언론사에 도착한 뒤 그 언론사의 robots.txt를 확인해 허용될 때만 정보를 읽고, 불허면 "미수집"으로 기록만 했다. 읽는 정보는 언론사가 검색·공유용으로 공개한 요약·제목·발행일뿐이며, **기사 본문 전체는 저장하지 않았다.** 기사 사이에는 3초 간격을 두고, 화면 없는 헤드리스 모드로 실행했다. 셀레니움이나 크롬을 쓸 수 없는 환경에서는 이 단계를 건너뛰고 헤드라인만으로 분석을 이어가도록 설계했다.

### 3.6 기사 분석과 가중치 계산 (Q3)

**감성 분석을 쓰지 않은 이유.** "석방되었다", "무기를 내려놓았다"는 감성상 긍정이지만, KYC에서 중요한 것은 분위기가 아니라 **인물이 어떤 역할로 어떤 사건에 등장했는가**다. 헤드라인(셀레니움으로 요약을 확보한 기사는 헤드라인 + 요약)마다 행위 유형(무장 활동, 혐오·선동, 사법·제재, 경제·자원, 무장해제·평화, 협상·중재, 억류·피해, 정치 활동, 발언)과 인물의 역할(행위 주체, 발언 주체, 언급 대상, 피해자)을 분류했다. 1차 분류는 프랑스어 키워드 규칙이며, 선택적으로 Claude API가 문맥을 판단하고, 검토자가 기사별로 수정할 수 있다.

**계산 구조.** 부정적 언론 판단의 세 축(관련성·신뢰성·중대성)에 절차 단계·최신성·역할을 더해 곱하고, 같은 사건을 묶은 뒤 독립 증거를 결합했다.

| 단계 | 방법 | 근거 |
|---|---|---|
| 기사 점수 | 동일인 확신도 × 중대성 × 절차 단계 × 출처 신뢰도 × 시간 감쇠 × 역할 | 곱셈 구조로 한 요소라도 약하면 전체가 낮아짐 |
| 동일인 확신도 | 확정 1.0, 동일인 후보 0.8 | 관련성 |
| 중대성 | 무장 활동 1.0, 혐오·선동 0.9, 사법·제재 0.8, 경제·자원 0.6 / 무장해제 0.6, 협상 0.5 (차감 방향) | 지정 사유와의 관련 정도 |
| 절차 단계 | 유죄 1.0, 기소·체포 0.8, 수사·의혹 0.6, 기타 0.7 | 의혹과 확정의 구분 |
| 출처 신뢰도 | 국제 통신사·공영 1.0, 현지 주요 매체 0.8, 기타 0.6 | Admiralty식 출처 등급 단순화 |
| 시간 감쇠 | 0.5^(경과 연수 / 3) | 반감기 3년 지수 감쇠 |
| 역할 | 행위 주체 1.0, 언급 0.6, 발언 0.4, 피해자 0 (가산 기준) | 가해자·피해자 구분 |
| 사건 묶기 | 같은 인물, 3일 이내, 헤드라인 유사도 0.4 이상 → 최고 점수만 | 받아쓰기 보도의 중복 계산 방지 |
| 결합 | 사건 점수를 noisy-OR로 결합 → 가산 증거 P+, 차감 증거 P− | 독립 증거의 체감 결합 |

> 최종 가중치 = 기본 치명도 + 0.15 × P+ − 0.05 × P− (B 하한 0.30, C 하한 0.05)
>
> A급은 계산하지 않고 1.00 고정. 해제·추가 지정 보도는 가중치가 아니라 재분류 신호로 따로 표시한다.

**제재 이행 관점의 방향 정의.** "가산"은 지정 사유가 이어지거나 회피 정황이 있어 **제재 유지·강화가 정당화되는 신호**, "차감"은 무장 해제·평화 활동처럼 **지정 사유와 반대 방향의 행보**를 뜻한다. 피해자로 등장하거나 일반 의정 활동인 기사는 조정하지 않는다.

### 3.7 데이터 품질 검증

| 발견된 문제 | 증상 | 조치 |
|---|---|---|
| GDELT 요청 제한 | 기사 0건과 요청 실패가 구분되지 않을 위험 | 본 수집 전 API 테스트, 실패 시 중단 |
| RSS 302 응답 | 기사가 있는데 0건으로 처리 | 리다이렉트 추적 |
| 검색어 어순 | 공식 명단 어순으로는 기사 미검색 | 두 어순 OR 검색 |
| 날짜 파싱 | 요일 접두어 때문에 날짜가 비어 최신순 정렬·시간 감쇠 왜곡 | 요일 제거 후 고정 형식 파싱 |
| 구글 뉴스 링크 | 자바스크립트 리다이렉트라 requests로 기사 페이지에 도달 불가 | 셀레니움으로 이동을 기다린 뒤 도착 주소 확인 |
| 헤드라인 정보 부족 | 헤드라인만으로는 사건 성격·인물 역할 판단 불가 | 언론사 공개 요약을 함께 분류 |
| 동의 페이지·로딩 실패 | 브라우저가 언론사 대신 동의 화면에서 멈춤 | "동의 페이지에서 멈춤"·"실패"로 구분 기록, 분류에서 제외 |
| 키워드 오탐 | "armée(군대)", "armes(무기)"가 무장 활동으로 분류 | "groupe armé(무장단체)"로 한정 |
| 동명이인 | 이름 단어만 겹친 후보 발생 | 생년월일 대조로 오탐 판정 |

모든 문제는 단계별 개수 확인과 테스트 실행으로 발견했으며, 결과가 0이거나 비정상일 때 그대로 결론으로 받아들이지 않았다.

## 4. 결과

### 4.1 Q1: 제재 대상 PEP

| 판정 | 인물 | 근거 | 목록 수 |
|---|---|---|---|
| 확정 | Bitakwira Bihona Hayi Justin | OpenSanctions ID 일치 | 7 |
| 동일인 가능성 높음 | Ndjabu Ngabu Floribert | "Floribert Ngabu" 포함 + 생년월일 일치 | 15 |
| 오탐 | Wagner Tshienda Tshienda Richard | "Richard Wagner"(이란 목록)와 단어만 겹침, 생년월일 불일치 | 1 |

**의원 560명 중 실제 제재 대상은 2명(0.36%)** 이다. Ndjabu는 OpenSanctions가 ID로 연결하지 않아 정확 일치나 ID 대조만으로는 찾을 수 없었던 인물로, 어순 무관 매칭과 생년월일 대조가 함께 작동해 발견됐다. 반대로 Wagner는 넓은 매칭이 만든 오탐으로, **이름만으로 판단했다면 무고한 의원이 이란 제재 대상으로 오인됐을 것**이다.

### 4.2 Q2: 제재 정도

**원지정 기관 기준 구성.**

| 인물 | 원지정 기관 | 목록 수 | 원지정 | 타국 이행 | 카테고리 |
|---|---|---|---|---|---|
| Ndjabu | UN 안보리 | 5 | 1 | 4 (호주·일본·남아공·아르헨티나) | A |
| Ndjabu | EU | 7 | 3 | 4 (벨기에·프랑스·모나코·스위스) | B·C |
| Ndjabu | 영국 | 1 | 1 | 0 | B |
| Ndjabu | 대만(수출통제)·우크라이나(FIU 목록) | 2 | 2 | 0 | B·C |
| Bitakwira | EU | 7 | 3 | 4 (벨기에·프랑스·모나코·스위스) | B·C |

| 인물 | A | B | C | 목록 합계 | 최종 카테고리 | 치명도 |
|---|---|---|---|---|---|---|
| Ndjabu Ngabu Floribert | 5 | 7 | 3 | 15 | A | 1.00 |
| Bitakwira Bihona Hayi Justin | 0 | 5 | 2 | 7 | B | 0.44 |

목록 수로는 15 대 7로 두 배 넘게 차이 나지만, 본질적 차이는 **UN(전 회원국 구속) 대 EU(지역 구속)** 이다. Ndjabu의 15개 목록 중 금융 제재로서 독립적인 결정은 UN·EU·영국 세 곳이고, Bitakwira의 7개는 사실상 EU 결정 하나에서 나왔다. 대만 수출통제와 우크라이나 FIU 목록은 금융거래 금지가 아니어서 위험 판단에 미치는 영향이 작다.

**기관 가정 민감도.**

| 기관 가정 | Ndjabu | Bitakwira |
|---|---|---|
| 기본 (EU·영국 연결 없음, 달러 결제 있음) | A | B |
| EU 연결 있음 | A | **A** |
| 영국 연결 있음 | A | B |
| 달러 결제 없음 | A | B |

Ndjabu는 어떤 가정에서도 A급이지만, Bitakwira는 기관에 EU 연결고리가 있으면 A급이 된다. **제재 위험은 사람의 속성이 아니라 사람과 기관의 관계로 정해진다.**

**판정 신뢰도와 위험 크기의 역전.** Bitakwira는 동일인 여부가 확정됐지만 위험은 B급이고, Ndjabu는 동일인 여부가 후보 단계지만 위험은 A급이다. 오판 시 대가가 가장 큰 A급 후보의 동일인 확인을 최우선으로 처리해야 한다.

### 4.3 Q3: 뉴스의 제재 이행 방향

| 인물 | 수집 기사 | 기사 분석 결과 | 방향 | 가중치 |
|---|---|---|---|---|
| Ndjabu Ngabu Floribert | 9건 | 무장 해제 협상 중재자·억류 피해자로 등장, 이투리 무장단체 출신 인물(토마 루방가)과 동시 출현 | 차감 방향(완화 신호), 가산 없음 | 1.00 유지 (A급 오버라이드) |
| Bitakwira Bihona Hayi Justin | 74건 | 행위 유형·역할 분류 후 독립 사건 기준으로 결합 | 실행 결과에 따름 | 0.44 → 실행 결과 (0.39~0.59 범위) |

**Ndjabu의 대표 기사.**

| 날짜 | 헤드라인 (번역) | 행위 유형 | 역할 | KYC상 의미 |
|---|---|---|---|---|
| 확인 필요 | Floribert Ndjabu : « Ils acceptent de déposer les armes » (그들이 무기를 내려놓기로 했다) | 무장해제·평화 | 발언자(중재) | 지정 사유와 반대 방향 행보, 동일인 확인 근거 |
| 2022-04-12 | 토마 루방가, 플로리베르 은자부와 군 장교 2명 석방 (Actualite.cd) | 억류·피해 | 피해자 | 위험 조정 없음, 동일인 확인 근거(강함) |

감성 분석이었다면 두 기사가 "긍정"으로 분류되어 위험을 낮추는 근거로 오용될 수 있었다. 실제 판단은 다르다. **UN 제재가 유효한 한 평화 활동 보도가 있어도 거래 금지는 바뀌지 않으며**, 바뀔 수 있는 것은 UN의 공식 해제뿐이다. 대신 무장단체 맥락과 동시 출현 인물은 UN 지정 인물 "Floribert Ngabu"의 경력과 맞아, 판정을 **"동일인 후보" → "동일인 확정 검토"** 로 올리는 근거가 된다. 평화 활동이 장기간 이어지면 해제 논의로 이어질 수 있으므로 해제 보도를 지속 감시한다.

**Bitakwira의 판정 규칙.** 74건이라는 기사 수는 반영하지 않는다. 지정 사유와 같은 행위(폭력 선동 등)의 행위 주체 보도나 회피 정황은 가산, 장기간의 화해 활동은 차감, 일반 의정 활동은 조정 없음이다. EU 제재 미연장·해제 보도는 C급 또는 해제로, UN·미국 추가 지정 보도는 A급으로 **재분류 검토**한다.

| 기사 분류 결과 | 조정 | 최종 가중치 | 은행 대응 |
|---|---|---|---|
| 대부분 일반 정치 활동 | 0 | 0.44 | B급 EDD 유지 |
| 최근 지정 사유 관련 행위의 행위 주체 보도 | +0.05 ~ +0.10 | 0.49 ~ 0.54 | EDD 강화, 검토 주기 단축 |
| 제재 회피 정황 보도 | +0.15 | 0.59 | 거래 제한, 관련 인물·법인 고위험 지정 |
| 화해 활동 장기 지속 | −0.05 | 0.39 | B급 하한(0.30) 이상에서 유지 |
| EU 제재 미연장·해제 보도 | 재분류 | C급 또는 해제 | 공식 관보 확인 후 등급 변경 |
| UN·미국 추가 지정 보도 | 재분류 | A급 | 공식 확인 즉시 거래 금지 |

**셀레니움으로 확인한 기사 페이지.** 인물별 최신 5건의 기사 페이지를 열어 언론사 공개 요약을 확보했다. 기사별 처리 결과(수집·robots.txt 불허·동의 페이지·실패)와 확보한 요약은 `drc_news_leads.csv`에 저장되며, 노트북의 실행 결과 셀에서 집계된다. 요약을 확보한 기사는 헤드라인만 있을 때보다 판단 근거가 분명해진다. 예를 들어 "○○를 둘러싼 논란 계속" 같은 헤드라인은 논란의 내용을 알 수 없지만, 요약에 "폭력 선동 혐의를 받고 있다"는 문장이 있으면 "혐오·선동 / 행위 주체"로 분류되어 가중치에 반영된다. 반대로 요약에 "민병대"라는 단어가 있어도 인물이 무장 해제를 전하는 중재자라면 무장 활동이 함께 잡히는 키워드 분류의 한계가 있어, 이런 기사는 검토 파일로 바로잡는다.

**기사 수와 위험의 역전.** 언론 노출은 B급인 Bitakwira(74건)가 A급인 Ndjabu(9건)보다 8배 많다. 기사 수는 위험이 아니라 **공인으로서의 노출량**에 가깝다는 점이 데이터로 확인된다.

## 5. KYC 실무 시사점

| 이번 분석 | 실무 대응 |
|---|---|
| OpenSanctions 명단 | 상용 워치리스트 (라이선스 필요) |
| 어순 무관 매칭 + 생년월일 | 스크리닝 엔진 + 얼럿 검토 (오탐 해소) |
| A/B/C 카테고리 | 거래 금지 오버라이드와 EDD 구분 |
| 기사 분류·가중치 | 부정적 언론 검토, 동일인 확인 |
| 30분 주기 모니터 | 지속 모니터링, 이벤트 기반 재스크리닝 |

실무는 국가 단위 명단이 아니라 **기관의 고객**에서 출발하며, PEP이면서 제재 대상이면 제재가 우선해 A급은 EDD가 아니라 거래 차단과 당국 보고로 이어진다. 기사만으로 동결하지 않으며, 항상 **기사 감지 → 공식 공시 확인 → 재스크리닝 → 조치 → 근거 기록** 순서를 따른다.

DR Congo 사례는 분쟁 이후 무장세력 지도자가 정치권에 편입되면서 PEP와 제재 대상이 겹칠 수 있음을 보여준다. 이런 고객의 EDD는 재산 출처를 분쟁 경제와 연결된 부문(광물 거래, 국경 교역)과 대조하는 방향으로 이루어져야 한다. 기사의 실무적 가치는 제재 대상 자체보다 **구속력 없는 제재(B급)의 위험 판단, 동일인 확인, 명단에 없는 가족·측근 식별, 그리고 제재 대상이 아닌 나머지 558명의 조기 경보**에 더 크다.

## 6. 한계

| 영역 | 한계 | 보완 방향 |
|---|---|---|
| 동일인 판단 | 이름·생년월일 기반 | 원 공시의 국적·신분증·지정 사유로 최종 대조 |
| 제재 명목·인적사항 | 상세 데이터에 없는 항목은 "정보 없음" | UN 1533 위원회 목록, EU 관보 등 원 공시 확인 |
| 라벨링 | 목록 이름 기준 분류 | 원 조치 내용 확인 |
| 가정값 | 카테고리 기준·AHP 판단·조정 폭은 분석자 가정 | 전문가 다수 판단의 기하평균, 민감도 분석 |
| 기사 분석 | 헤드라인 + 언론사 공개 요약 기반 키워드 분류 | 검토자 수정, 언어 모델 문맥 판단 |
| 기사 페이지 확인 | 최신 5건만, robots.txt 불허·동의 페이지·로딩 실패 기사는 요약 없음 | 확인 범위 확대, 실패 기사 수동 확인 |
| 뉴스 범위 | 제재 인물 2명만, RSS는 과거 기사 포함 | 558명 확장, 연도별 분포 확인 |
| 날짜 | Ndjabu 무장 해제 기사 날짜 미확인 | 본문 확인 |
| 대조 범위 | "제재" 데이터만 포함 | 입국 금지·부패 지정 등 다른 위험 목록 추가 |

PEP 분류, 제재 여부, 기사 수와 가중치는 그 자체로 개인의 위법을 단정하지 않는다.

## 7. 윤리 점검

OpenSanctions는 robots.txt 허용 경로만 10초 간격(Crawl-delay)으로 접근했고, 금지 경로는 공식 파일로 대체했다. GDELT 429에는 IP 변경 같은 우회 없이 수집을 중단하고 설계를 바꿨다. 뉴스 목록은 자동 수신용 공개 형식인 RSS로 3초 간격 수집했다. 셀레니움으로 연 기사 페이지는 **언론사별 robots.txt가 허용할 때만** 읽었고, 언론사가 공개한 요약·제목·발행일만 저장했으며 **기사 본문 전체는 저장하지 않았다.** 기사 사이에는 3초 간격을 두었고, 차단이나 동의 화면을 우회하지 않았다. 데이터는 비상업 학습용(CC BY-NC)으로만 쓰고, 명단·기사 CSV는 `.gitignore`로 저장소에서 제외했다.

## 8. 결론

DR Congo 국회의원 560명 중 **2명이 제재 대상 PEP**이며, 두 사람의 위험은 같지 않다. **Ndjabu Ngabu Floribert는 UN 기반 A급(치명도 1.00)** 으로 거래 금지 대상이고, **Bitakwira Bihona Hayi Justin은 EU 기반 B급(치명도 0.44)** 으로 강화된 확인 대상이다. 뉴스는 Ndjabu에게는 위험을 바꾸지 않고 동일인 확인 근거로 작용했으며, Bitakwira에게는 기사 내용에 따라 정해진 범위 안에서 가중치를 조정하는 동향 지표로 작용한다.

분석 전반에서 확인된 원칙은 네 가지다. 공식 명단의 이름 형식을 이해한 **매칭 설계가 결과를 결정**하며, 탐지 범위를 넓힐수록 **두 번째 식별자로 반드시 걸러야** 한다. "제재 대상"은 하나의 범주가 아니라 **원지정 기관·조치 종류·기관과의 관계**로 나뉘며, 목록 수는 위험의 크기를 그대로 반영하지 않는다. 기사는 **분위기가 아니라 역할로** 읽어야 하며, 기사 수는 위험이 아니라 노출량이다. 마지막으로 0건과 이상값은 결론이 아니라 **점검의 대상**이며, 수집 제한은 우회가 아니라 설계 변경으로 해결한다.

---

## 부록. 목록별 라벨링 전체

| 인물 | 제재 목록 | 원지정 기관 | 구분 | 조치 | 카테고리 |
|---|---|---|---|---|---|
| Ndjabu | UN Security Council Consolidated Sanctions | UN 안보리 | 원지정 | 자산동결·여행금지 | A |
| Ndjabu | Australian Sanctions Consolidated List | UN 안보리 | 이행(호주) | 자산동결 | A |
| Ndjabu | Japan Economic Sanctions and List of Eligible People | UN 안보리 | 이행(일본) | 자산동결 | A |
| Ndjabu | South Africa Targeted Financial Sanctions | UN 안보리 | 이행(남아공) | 자산동결 | A |
| Ndjabu | Argentina RePET Sanctions | UN 안보리 | 이행(아르헨티나) | 자산동결 | A |
| Ndjabu | EU Financial Sanctions Files (FSF) | EU | 원지정 | 자산동결 | B |
| Ndjabu | Belgian Financial Sanctions | EU | 이행(벨기에) | 자산동결 | B |
| Ndjabu | French National Asset Freezing System | EU | 이행(프랑스) | 자산동결 | B |
| Ndjabu | Monaco National Fund Freezing List | EU | 이행(모나코) | 자산동결 | B |
| Ndjabu | Swiss SECO Sanctions/Embargoes | EU | 이행(스위스) | 자산동결 | B |
| Ndjabu | UK FCDO Sanctions List | 영국 | 원지정 | 자산동결 | B |
| Ndjabu | Taiwan Strategic High-Tech Commodities Entity List | 대만 | 독자 | 수출통제 | B |
| Ndjabu | EU Consolidated Travel Bans | EU | 원지정 | 여행금지 | C |
| Ndjabu | EU Council Official Journal Sanctioned Entities | EU | 원지정 | 지정 근거(관보) | C |
| Ndjabu | Ukraine SFMS Blacklist | 우크라이나 | 독자 | FIU 감시목록 | C |
| Bitakwira | EU Financial Sanctions Files (FSF) | EU | 원지정 | 자산동결 | B |
| Bitakwira | Belgian Financial Sanctions | EU | 이행(벨기에) | 자산동결 | B |
| Bitakwira | French National Asset Freezing System | EU | 이행(프랑스) | 자산동결 | B |
| Bitakwira | Monaco National Fund Freezing List | EU | 이행(모나코) | 자산동결 | B |
| Bitakwira | Swiss SECO Sanctions/Embargoes | EU | 이행(스위스) | 자산동결 | B |
| Bitakwira | EU Consolidated Travel Bans | EU | 원지정 | 여행금지 | C |
| Bitakwira | EU Council Official Journal Sanctioned Entities | EU | 원지정 | 지정 근거(관보) | C |